# Historical research notebook

Sanitized source archive, not a validated current model. Saved outputs were removed. Data, credentials and model weights are not included. Path placeholders are not automatically expanded. See docs/CODE_CHANGELOG.md.


# MLB GPT Prompt v11 - 백테스트 모드 (2025 정규시즌)
- v11 동일 구조 / 날짜·경로만 2025 백테스트용으로 교체
- 태그 체계 포함 / 검증 목적

In [ ]:
# =============================================
# MLB GPT 프롬프트 자동 생성기 - 백테스트 v1
# 사용 방법:
#   1. BT_DATE 설정 (분석할 날짜)
#   2. 셀 순서대로 실행
#   3. 출력된 프롬프트 GPT/Claude에 붙여넣기
# =============================================


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import requests
import time
from datetime import datetime, timedelta
from io import StringIO

MLB_PATH = "<DRIVE_ROOT>/MLB데이터"


In [ ]:
# =============================================
# 백테스트 날짜 설정 (여기만 바꾸면 됨)
# =============================================
BT_DATE = "2025-04-01"   # ← 분석할 날짜 변경

DATE   = BT_DATE
TODAY  = BT_DATE.replace("-", "")
BT_PATH = f"{MLB_PATH}/2025/백테스트_라인업"
MANUAL_RUN_TOTAL_BASELINES = {}

# 필요 파일 체크
needed = {
    "경기목록":   f"{BT_PATH}/mlb_2025_백테스트_경기목록.csv",
    "라인업":     f"{BT_PATH}/mlb_2025_백테스트_라인업.csv",
    "날씨":       f"{BT_PATH}/mlb_2025_백테스트_날씨.csv",
    "불펜소모도":  f"{BT_PATH}/mlb_2025_백테스트_불펜소모도.csv",
    "총득점기준점":     f"{BT_PATH}/mlb_2025_정규시즌_총득점기준점.csv",
    "IL현황":     f"{BT_PATH}/mlb_2025_백테스트_IL현황.csv",
    "마이너타자":  f"{MLB_PATH}/2025/mlb_2025_마이너성적_타자.csv",
    "마이너투수":  f"{MLB_PATH}/2025/mlb_2025_마이너성적_투수.csv",
}
for label, path in needed.items():
    print(f"{'✅' if os.path.exists(path) else '❌'} {label}: {os.path.basename(path)}")
print(f"\n백테스트 날짜: {BT_DATE}")


In [ ]:
# =============================================
# 데이터 로드 (백테스트 v1)
# =============================================

print("데이터 로드 중...")

def load_csvs(paths):
    dfs = []
    for p in paths:
        if os.path.exists(p):
            try:
                df = pd.read_csv(p)
                if not df.empty:
                    dfs.append(df)
            except:
                pass
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# 경기결과 (2023~2025) - 2026 제외
result_paths = []
for year in [2023, 2024, 2025]:
    for series in ["시범경기", "정규시즌"]:
        result_paths.append(f"{MLB_PATH}/{year}/mlb_{year}_{series}_경기결과.csv")
df_results = load_csvs(result_paths)
df_results["날짜"] = pd.to_datetime(df_results["날짜"])

# 투수기록 (2023~2025)
pitcher_paths = []
for year in [2023, 2024, 2025]:
    for series in ["시범경기", "정규시즌"]:
        pitcher_paths.append(f"{MLB_PATH}/{year}/mlb_{year}_{series}_투수개인기록.csv")
df_pitchers = load_csvs(pitcher_paths)

# 선수시즌기록
player_pit_paths = [f"{MLB_PATH}/{y}/mlb_{y}_선수시즌기록_투수.csv" for y in [2023, 2024, 2025]]
df_player_pit = load_csvs(player_pit_paths)
player_hit_paths = [f"{MLB_PATH}/{y}/mlb_{y}_선수시즌기록_타자.csv" for y in [2023, 2024, 2025]]
df_player_hit = load_csvs(player_hit_paths)

# 마이너 성적 (2025 백테스트용)
_minor_hit_path = f"{MLB_PATH}/2025/mlb_2025_마이너성적_타자.csv"
if os.path.exists(_minor_hit_path):
    df_minor_hit = pd.read_csv(_minor_hit_path)
    if df_minor_hit.columns[0] != "선수명":
        df_minor_hit = pd.read_csv(_minor_hit_path, header=None,
            names=["선수명","선수ID","avg","ops","homeRuns","plateAppearances","hits","atBats","출처","팀"])
    df_minor_hit = df_minor_hit.fillna("")
else:
    df_minor_hit = pd.DataFrame()

_minor_pit_path = f"{MLB_PATH}/2025/mlb_2025_마이너성적_투수.csv"
df_minor_pit = pd.read_csv(_minor_pit_path).fillna("") if os.path.exists(_minor_pit_path) else pd.DataFrame()

# 매치업 누적 (2025 없으면 빈 DataFrame)
df_matchup = pd.DataFrame()

# Statcast
sc_pit_paths = [f"{MLB_PATH}/{y}/mlb_{y}_statcast_투수.csv" for y in [2023,2024,2025]]
sc_bat_paths = [f"{MLB_PATH}/{y}/mlb_{y}_statcast_타자.csv" for y in [2023,2024,2025]]
df_sc_pit = load_csvs(sc_pit_paths)
df_sc_bat = load_csvs(sc_bat_paths)

# 심판 (2023~2025)
umpire_paths = [f"{MLB_PATH}/{y}/mlb_{y}_심판.csv" for y in [2023, 2024, 2025]]
df_umpires = load_csvs(umpire_paths)

# 트랜잭션 (2023~2025)
trans_paths = [f"{MLB_PATH}/{y}/mlb_{y}_트랜잭션.csv" for y in [2023, 2024, 2025]]
df_trans = load_csvs(trans_paths)

# 타자기록 (2023~2025)
bat_paths = []
for year in [2023, 2024, 2025]:
    for series in ["시범경기", "정규시즌"]:
        bat_paths.append(f"{MLB_PATH}/{year}/mlb_{year}_{series}_타자개인기록.csv")
df_batters = load_csvs(bat_paths)

# 피치데이터 (2023~2025)
pitch_paths = [f"{MLB_PATH}/{y}/mlb_{y}_정규시즌_피치데이터.csv" for y in [2023,2024,2025]]
df_pitch = load_csvs([p for p in pitch_paths if os.path.exists(p)])

# 키플레이어 (2023~2025)
df_kp = pd.DataFrame()
for year in [2023, 2024, 2025]:
    for series in ["시범경기", "정규시즌"]:
        kp_path = f"{MLB_PATH}/{year}/mlb_{year}_{series}_키플레이어.csv"
        if os.path.exists(kp_path):
            try:
                df_tmp = pd.read_csv(kp_path)
                if "게임스코어" in df_tmp.columns and "타입" in df_tmp.columns:
                    df_kp = pd.concat([df_kp, df_tmp], ignore_index=True)
            except:
                pass

print(f"경기결과: {len(df_results)}경기 | 투수기록: {len(df_pitchers)}행 | 타자기록: {len(df_batters)}행")
print(f"Statcast 투수: {len(df_sc_pit)} | Statcast 타자: {len(df_sc_bat)}")
print(f"마이너 타자: {len(df_minor_hit)} | 마이너 투수: {len(df_minor_pit)} | 매치업: {len(df_matchup)}")

# ── 백테스트 날짜별 파일 로드 ──
_bt_date_str = pd.to_datetime(BT_DATE).strftime("%Y%m%d")

# 총득점 기준점 (날짜 필터)
_run_total_all = pd.read_csv(f"{BT_PATH}/mlb_2025_정규시즌_총득점기준점.csv") if os.path.exists(f"{BT_PATH}/mlb_2025_정규시즌_총득점기준점.csv") else pd.DataFrame()
if not _run_total_all.empty:
    _run_total_all["날짜_str"] = pd.to_datetime(_run_total_all["date"]).dt.strftime("%Y-%m-%d")
    df_run_total_baselines_raw = _run_total_all[_run_total_all["날짜_str"] == BT_DATE].copy()
    # v11 형식으로 변환 (원정팀/홈팀/총득점기준점)
    if not df_run_total_baselines_raw.empty:
        df_run_total_baselines = df_run_total_baselines_raw.rename(columns={"home_team":"홈팀","away_team":"원정팀","run_total_baseline":"총득점기준점"})
    else:
        df_run_total_baselines = pd.DataFrame()
else:
    df_run_total_baselines = pd.DataFrame()

# 날씨 (날짜 필터)
_weather_all = pd.read_csv(f"{BT_PATH}/mlb_2025_백테스트_날씨.csv") if os.path.exists(f"{BT_PATH}/mlb_2025_백테스트_날씨.csv") else pd.DataFrame()
if not _weather_all.empty:
    _weather_all["날짜"] = pd.to_datetime(_weather_all["날짜"]).dt.strftime("%Y-%m-%d")
    df_weather = _weather_all[_weather_all["날짜"] == BT_DATE].copy()
else:
    df_weather = pd.DataFrame()

print(f"RUN_TOTAL:{len(df_run_total_baselines)} 날씨:{len(df_weather)}")

# 선발시범성적 (2025 시범경기 투수기록에서 계산)
_spring_raw = pd.read_csv(f"{MLB_PATH}/2025/mlb_2025_시범경기_투수개인기록.csv") if os.path.exists(f"{MLB_PATH}/2025/mlb_2025_시범경기_투수개인기록.csv") else pd.DataFrame()
if not _spring_raw.empty:
    def _ip_f(x):
        try:
            parts = str(x).split(".")
            return int(parts[0]) + int(parts[1])/3 if len(parts)>1 else float(x)
        except: return 0.0
    _spring_raw["IP_val"] = _spring_raw["이닝"].apply(_ip_f)
    _spring_grp = _spring_raw.groupby("선수명").agg(
        spring_등판수=("gamePk","count"),
        spring_이닝=("IP_val","sum"),
        총자책=("자책","sum")
    ).reset_index()
    _spring_grp["spring_ERA"] = (_spring_grp["총자책"]*9/_spring_grp["spring_이닝"]).round(2).where(_spring_grp["spring_이닝"]>0, 0.0)
    _spring_grp["spring_이닝"] = _spring_grp["spring_이닝"].round(1)
    df_spring_stats = _spring_grp[["선수명","spring_등판수","spring_이닝","spring_ERA"]]
    print(f"선발시범성적 로드: {len(df_spring_stats)}명")
else:
    df_spring_stats = pd.DataFrame()

# IL현황 (날짜+gamePk 필터)
_il_all = pd.read_csv(f"{BT_PATH}/mlb_2025_백테스트_IL현황.csv") if os.path.exists(f"{BT_PATH}/mlb_2025_백테스트_IL현황.csv") else pd.DataFrame()
if not _il_all.empty:
    _il_all["날짜"] = pd.to_datetime(_il_all["날짜"]).dt.strftime("%Y-%m-%d")
    df_il_today = _il_all[_il_all["날짜"] == BT_DATE].copy()
    print(f"IL현황 로드: {len(df_il_today)}명")
else:
    df_il_today = pd.DataFrame()
    print("IL현황 파일 없음")

# 불펜소모도 (날짜 필터)
_bp_all = pd.read_csv(f"{BT_PATH}/mlb_2025_백테스트_불펜소모도.csv") if os.path.exists(f"{BT_PATH}/mlb_2025_백테스트_불펜소모도.csv") else pd.DataFrame()
if not _bp_all.empty:
    _bp_all["날짜"] = pd.to_datetime(_bp_all["날짜"]).dt.strftime("%Y-%m-%d")
    df_bp_detail_today = _bp_all[_bp_all["날짜"] == BT_DATE].copy()
    print(f"불펜소모도 로드: {len(df_bp_detail_today)}명")
else:
    df_bp_detail_today = pd.DataFrame()

# 전년도 투수기록 (예상이닝 계산용) - 2024만 사용 (2025 백테스트이므로)
def _ip_to_float(x):
    try:
        parts = str(x).split(".")
        return int(parts[0]) + int(parts[1])/3 if len(parts)>1 else float(x)
    except: return 0.0

_prev_paths = [f"{MLB_PATH}/2024/mlb_2024_정규시즌_투수개인기록.csv"]
_prev_dfs = [pd.read_csv(p) for p in _prev_paths if os.path.exists(p)]
df_pit_prev = pd.concat(_prev_dfs, ignore_index=True) if _prev_dfs else pd.DataFrame()
if not df_pit_prev.empty:
    df_pit_prev["IP_val"] = df_pit_prev["이닝"].apply(_ip_to_float)
    print(f"전년도 투수기록 로드: {len(df_pit_prev)}행")


In [ ]:

def _clean_ops(val):
    try:
        if str(val).lower() in ["nan", "none", "", "n/a"]: return "N/A"
        f = float(val)
        if f > 3.0 or f < 0: return "N/A"
        return val
    except: return "N/A"

def _clean_avg(val):
    try:
        if str(val).lower() in ["nan", "none", "", "n/a"]: return "N/A"
        f = float(val)
        return "N/A" if f > 1.0 or f < 0 else val
    except: return "N/A"

def _clean_source(src):
    # age 숫자가 출처 자리에 들어온 경우 필터링
    if not src: return "데이터없음"
    try:
        float(src)  # 순수 숫자면 age 오류
        return "데이터없음"
    except:
        return src
# =============================================
# 기존 피처 함수 (v2와 동일)
# =============================================

def get_recent_team_stats(df, team, before_date, n=10):
    before_date = pd.to_datetime(before_date)
    home = df[(df["홈팀"] == team) & (df["날짜"] < before_date)].copy()
    away = df[(df["원정팀"] == team) & (df["날짜"] < before_date)].copy()
    home["득점"] = home["홈팀득점"]; home["실점"] = home["원정팀득점"]; home["승"] = (home["홈팀결과"] == "승").astype(int)
    away["득점"] = away["원정팀득점"]; away["실점"] = away["홈팀득점"]; away["승"] = (away["원정팀결과"] == "승").astype(int)
    all_games = pd.concat([home[["날짜","득점","실점","승","총득점"]], away[["날짜","득점","실점","승","총득점"]]]).sort_values("날짜").tail(n)
    if all_games.empty:
        return {"경기수":0,"승":0,"패":0,"승률":0,"평균득점":0,"평균실점":0,"평균총득점":0}
    return {"경기수":len(all_games),"승":int(all_games["승"].sum()),"패":int(len(all_games)-all_games["승"].sum()),
            "승률":round(all_games["승"].mean(),3),"평균득점":round(all_games["득점"].mean(),2),
            "평균실점":round(all_games["실점"].mean(),2),"평균총득점":round(all_games["총득점"].mean(),2)}

def get_home_away_stats(df, team, before_date, n=20):
    before_date = pd.to_datetime(before_date)
    home = df[(df["홈팀"]==team)&(df["날짜"]<before_date)].tail(n)
    away = df[(df["원정팀"]==team)&(df["날짜"]<before_date)].tail(n)
    return {"홈승률":round((home["홈팀결과"]=="승").mean(),3) if len(home)>0 else 0,
            "원정승률":round((away["원정팀결과"]=="승").mean(),3) if len(away)>0 else 0}

def get_head_to_head(df, home_team, away_team, n=30):
    h2h = df[((df["홈팀"]==home_team)&(df["원정팀"]==away_team))|((df["홈팀"]==away_team)&(df["원정팀"]==home_team))].tail(n)
    if h2h.empty: return None
    hw = len(h2h[((h2h["홈팀"]==home_team)&(h2h["홈팀결과"]=="승"))|((h2h["원정팀"]==home_team)&(h2h["원정팀결과"]=="승"))])
    return {"총경기":len(h2h),"홈팀승":hw,"원정팀승":len(h2h)-hw,
            "평균총득점":round(h2h["총득점"].mean(),2),"평균점수차":round(h2h["점수차"].mean(),2)}

def get_park_factor(df, stadium):
    reg = df[df["시리즈"]=="정규시즌"]
    overall_avg = reg["총득점"].mean()
    stadium_avg = reg[reg["구장"]==stadium]["총득점"].mean()
    if pd.isna(stadium_avg): return 100.0
    return round(stadium_avg/overall_avg*100,1)

def ip_to_float(x):
    try:
        parts = str(x).split(".")
        return int(parts[0]) + int(parts[1])/3 if len(parts)>1 else float(x)
    except: return 0.0

def get_pitcher_last_n(df_pit, df_res, pitcher_name, before_date, n=3):
    # [수정] 시범경기와 정규시즌 완전 분리
    if not pitcher_name or pitcher_name=="미정": return {}
    try:
        pit = df_pit[(df_pit["선수명"]==pitcher_name)&(df_pit["이닝"].astype(str)!="0.0")].copy()
        if "날짜" in pit.columns: pit = pit.drop(columns=["날짜"])
        pit = pit.merge(df_res[["gamePk","날짜","시리즈"]], on="gamePk", how="left")
        pit["날짜"] = pd.to_datetime(pit["날짜"])
        pit = pit[pit["날짜"]<pd.to_datetime(before_date)].sort_values("날짜",ascending=False)
        if pit.empty: return {}
        # 시범경기 우선, 없으면 정규시즌
        spring = pit[pit["시리즈"].str.contains("시범", na=False)].head(n)
        regular = pit[pit["시리즈"]=="정규시즌"].head(n)
        if not spring.empty:
            pit_use = spring; context = "시범경기(2026)"
        elif not regular.empty:
            pit_use = regular; context = "정규시즌(2025)"
        else:
            return {}
        return {"경기수":len(pit_use),"평균이닝":round(pit_use["이닝"].apply(ip_to_float).mean(),2),
                "평균자책":round(pit_use["자책"].mean(),2),"평균삼진":round(pit_use["삼진"].mean(),1),
                "출처":context}
    except: return {}

def get_starter_full_stats(df_player_pit, pitcher_name, season=2025):
    if not pitcher_name or pitcher_name=="미정": return {}
    row = df_player_pit[(df_player_pit["선수명"]==pitcher_name)&(df_player_pit["시즌"]==season)]
    if not row.empty:
        r = row.iloc[0]
        return {"출처":f"MLB {season}시즌","ERA":r.get("era",""),"WHIP":r.get("whip",""),
                "K/9":r.get("strikeoutsPer9Inn",""),"BB/9":r.get("walksPer9Inn",""),
                "승":r.get("wins",0),"패":r.get("losses",0),"이닝":r.get("inningsPitched","")}
    if "df_minor_pit" in dir() and df_minor_pit is not None and not df_minor_pit.empty:
        minor_row = df_minor_pit[df_minor_pit["선수명"]==pitcher_name]
        if not minor_row.empty:
            r = minor_row.iloc[0]
            return {"출처":r.get("출처","마이너"),"ERA":r.get("era",""),"WHIP":r.get("whip",""),
                    "K/9":r.get("strikeoutsPer9Inn",""),"BB/9":r.get("walksPer9Inn",""),
                    "승":r.get("wins",0),"패":r.get("losses",0),"이닝":r.get("inningsPitched",""),"팀":r.get("팀","")}
    try:
        people = requests.get("https://statsapi.mlb.com/api/v1/people/search",params={"names":pitcher_name},timeout=10).json().get("people",[])
        if not people: return {"출처":"데이터없음"}
        player_id = next((p["id"] for p in people if p.get("fullName","").lower()==pitcher_name.lower()), people[0]["id"])
        for sport_id,sport_name in [(11,"AAA"),(12,"AA"),(13,"A+"),(14,"A"),(16,"루키")]:
            res = requests.get(f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats",
                params={"stats":"season","season":season,"group":"pitching","sportId":sport_id},timeout=10).json()
            splits = res.get("stats",[{}])[0].get("splits",[])
            if splits:
                s = splits[0]["stat"]
                return {"출처":f"{sport_name} {season}시즌","ERA":s.get("era",""),"WHIP":s.get("whip",""),
                        "K/9":s.get("strikeoutsPer9Inn",""),"BB/9":s.get("walksPer9Inn",""),
                        "승":s.get("wins",0),"패":s.get("losses",0),"이닝":s.get("inningsPitched",""),
                        "팀":splits[0].get("team",{}).get("name","")}
            time.sleep(0.1)
        for prev_year in [season-1,season-2]:
            for sport_id,sport_name in [(1,"MLB"),(11,"AAA"),(12,"AA"),(13,"A+"),(14,"A")]:
                res = requests.get(f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats",
                    params={"stats":"season","season":prev_year,"group":"pitching","sportId":sport_id},timeout=10).json()
                splits = res.get("stats",[{}])[0].get("splits",[])
                if splits:
                    s = splits[0]["stat"]
                    league = splits[0].get("team",{}).get("name","")
                    return {"출처":f"{sport_name} {prev_year}시즌{f' ({league})' if sport_name!='MLB' else ''}",
                            "ERA":s.get("era",""),"WHIP":s.get("whip",""),"K/9":s.get("strikeoutsPer9Inn",""),
                            "BB/9":s.get("walksPer9Inn",""),"승":s.get("wins",0),"패":s.get("losses",0),"이닝":s.get("inningsPitched","")}
                time.sleep(0.1)
    except: pass
    return {"출처":"데이터없음"}

def get_batter_full_stats(df_player_hit, batter_name, season=2025, df_minor_hit=None, save_cache=True):
    if not batter_name: return {}
    row = df_player_hit[(df_player_hit["선수명"]==batter_name)&(df_player_hit["시즌"]==season)]
    if not row.empty:
        r=row.iloc[0]
        return {"출처":f"MLB {season}시즌","AVG":r.get("avg",""),"OPS":r.get("ops",""),"홈런":r.get("homeRuns",0),"타석":r.get("plateAppearances",0),"_ops_fix":True}
    if df_minor_hit is not None and not df_minor_hit.empty:
        mr = df_minor_hit[df_minor_hit["선수명"]==batter_name]
        if not mr.empty:
            r=mr.iloc[0]
            return {"출처":r.get("출처","마이너"),"AVG":r.get("avg",""),"OPS":r.get("ops",""),"홈런":r.get("homeRuns",0),"타석":r.get("plateAppearances",0),"팀":r.get("팀","")}
    try:
        people = requests.get("https://statsapi.mlb.com/api/v1/people/search",params={"names":batter_name},timeout=10).json().get("people",[])
        if not people: return {"출처":"데이터없음"}
        player_id = next((p["id"] for p in people if p.get("fullName","").lower()==batter_name.lower()), people[0]["id"])
        for sport_id,sport_name in [(11,"AAA"),(12,"AA"),(13,"A+"),(14,"A"),(16,"루키")]:
            res = requests.get(f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats",
                params={"stats":"season","season":season,"group":"hitting","sportId":sport_id},timeout=10).json()
            splits = res.get("stats",[{}])[0].get("splits",[])
            if splits:
                s=splits[0]["stat"]; team=splits[0].get("team",{}).get("name","")
                result={"출처":f"{sport_name} {season}시즌","AVG":s.get("avg",""),"OPS":s.get("ops",""),
                        "홈런":s.get("homeRuns",0),"타석":s.get("plateAppearances",0),"팀":team}
                if save_cache and df_minor_hit is not None:
                    # 필요한 컬럼만 저장 (age 등 불필요 컬럼 제외) + 중복 방지
                    _existing = df_minor_hit[df_minor_hit["선수명"]==batter_name] if not df_minor_hit.empty else pd.DataFrame()
                    if _existing.empty:
                        new_row={"선수명":batter_name,"선수ID":player_id,
                                 "avg":s.get("avg",""),"ops":s.get("ops",""),
                                 "homeRuns":s.get("homeRuns",0),"plateAppearances":s.get("plateAppearances",0),
                                 "hits":s.get("hits",0),"atBats":s.get("atBats",0),
                                 "출처":f"{sport_name} {season}시즌","팀":team}
                        pd.DataFrame([new_row]).to_csv(f"{MLB_PATH}/2026/mlb_2026_마이너성적_타자.csv",mode="a",header=False,index=False,encoding="utf-8-sig")
                return result
            time.sleep(0.1)
        for prev_year in [season-1,season-2]:
            for sport_id,sport_name in [(1,"MLB"),(11,"AAA"),(12,"AA"),(13,"A+"),(14,"A")]:
                res = requests.get(f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats",
                    params={"stats":"season","season":prev_year,"group":"hitting","sportId":sport_id},timeout=10).json()
                splits = res.get("stats",[{}])[0].get("splits",[])
                if splits:
                    s=splits[0]["stat"]
                    return {"출처":f"{sport_name} {prev_year}시즌","AVG":s.get("avg",""),"OPS":s.get("ops",""),"홈런":s.get("homeRuns",0),"타석":s.get("plateAppearances",0)}
                time.sleep(0.1)
    except: pass
    return {"출처":"데이터없음"}

def get_matchup_stats(df_matchup, df_pitch, batter_name, pitcher_name, pitcher_hand, batter_id=None):
    if not batter_name or not pitcher_name or pitcher_name=="미정": return {}
    if not df_matchup.empty:
        row = df_matchup[(df_matchup["타자"]==batter_name)&(df_matchup["투수"]==pitcher_name)]
        if not row.empty:
            r=row.iloc[0]
            return {"출처":"직접대결","타석":int(r.get("타석",0)),"AVG":r.get("AVG",""),"홈런":int(r.get("홈런",0))}
    if batter_id:
        try:
            res = requests.get(f"https://statsapi.mlb.com/api/v1/people/{batter_id}/stats",
                params={"stats":"careerStatSplits","group":"hitting","sitCodes":"vr,vl"},timeout=10).json()
            splits = res.get("stats",[{}])[0].get("splits",[])
            code = "vr" if pitcher_hand=="R" else "vl"
            for s in splits:
                if s.get("split",{}).get("code")==code:
                    stat=s["stat"]
                    return {"출처":f"커리어 vs {'우투' if pitcher_hand=='R' else '좌투'}",
                            "타석":stat.get("plateAppearances",0),"AVG":stat.get("avg",""),"OPS":stat.get("ops","")}
        except: pass
    return {"출처":"데이터없음"}

_pitcher_hand_cache = {}
def get_pitcher_hand(pitcher_name):
    if pitcher_name in _pitcher_hand_cache: return _pitcher_hand_cache[pitcher_name]
    try:
        people = requests.get("https://statsapi.mlb.com/api/v1/people/search",params={"names":pitcher_name},timeout=10).json().get("people",[])
        if people:
            hand=people[0].get("pitchHand",{}).get("code","R"); _pitcher_hand_cache[pitcher_name]=hand; return hand
    except: pass
    return "R"

def get_pitcher_pitch_stats(df_pitch, pitcher_name, min_pitches=20):
    if df_pitch.empty or not pitcher_name or pitcher_name=="미정": return {}
    pit = df_pitch[df_pitch["투수"]==pitcher_name].copy()
    if pit.empty: return {}
    in_play_events=["Single","Double","Triple","Home Run","Groundout","Flyout","Lineout","Pop Out","Field Error","Double Play"]
    hit_events=["Single","Double","Triple","Home Run"]
    swing_results=["Swinging Strike","In play, out(s)","In play, no out","In play, runs","Foul","Swinging Strike (Blocked)"]
    result={}
    for pitch_type,group in pit.groupby("구종"):
        if len(group)<min_pitches: continue
        swings=group[group["피치결과"].isin(swing_results)]; whiffs=group[group["피치결과"].isin(["Swinging Strike","Swinging Strike (Blocked)"])]
        in_play=group[group["타석결과"].isin(in_play_events)]; hits=group[group["타석결과"].isin(hit_events)]
        result[pitch_type]={"사용률":round(len(group)/len(pit)*100,1),"평균구속":round(group["구속"].mean(),1),
            "헛스윙률":round(len(whiffs)/len(swings)*100,1) if len(swings)>0 else 0,
            "피안타율":round(len(hits)/len(in_play),3) if len(in_play)>0 else 0}
    return result

def get_team_offense_pattern(df_results, df_batters, team, before_date, n=20):
    before_date=pd.to_datetime(before_date)
    home=df_results[(df_results["홈팀"]==team)&(df_results["날짜"]<before_date)].tail(n)
    away=df_results[(df_results["원정팀"]==team)&(df_results["날짜"]<before_date)].tail(n)
    recent_pks=pd.concat([home,away])["gamePk"].tolist()
    if not recent_pks: return {}
    _bat=df_batters[(df_batters["팀"]==team)&(df_batters["gamePk"].isin(recent_pks))].copy()
    if "날짜" in _bat.columns: _bat = _bat.drop(columns=["날짜"])
    bat=_bat
    if bat.empty: return {}
    all_scores=pd.concat([home["홈팀득점"],away["원정팀득점"]]); total_runs=all_scores.sum()
    total_hr=bat["홈런"].sum(); hr_runs=total_hr*1.4; total_h=bat["안타"].sum(); total_ab=bat["타수"].sum()
    return {"평균득점":round(all_scores.mean(),2),"홈런의존도":round(hr_runs/total_runs*100,1) if total_runs>0 else 0,
            "팀타율":round(total_h/total_ab,3) if total_ab>0 else 0,
            "득점유형":"홈런의존" if total_runs>0 and hr_runs/total_runs>0.35 else "출루의존"}

def get_pitcher_park_stats(df_pitchers, df_results, pitcher_name, stadium):
    if not pitcher_name or pitcher_name=="미정": return {}
    park_pks=df_results[df_results["구장"]==stadium]["gamePk"].tolist()
    if not park_pks: return {}
    pit=df_pitchers[(df_pitchers["선수명"]==pitcher_name)&(df_pitchers["gamePk"].isin(park_pks))]
    if pit.empty: return {}
    ip_vals=pit["이닝"].apply(ip_to_float); total_ip=ip_vals.sum()
    return {"등판수":len(pit),"ERA":round((pit["자책"].sum()*9)/total_ip,2) if total_ip>0 else 0,
            "WHIP":round((pit["피안타"].sum()+pit["볼넷"].sum())/total_ip,2) if total_ip>0 else 0}

def get_pitcher_home_away(df_pit, pitcher_name):
    # [수정] 이적 투수 처리: 현재팀 15경기+ 이면 현재팀만 사용
    pit=df_pit[df_pit["선수명"]==pitcher_name].copy()
    if pit.empty: return {}
    teams = pit["팀"].unique()
    if len(teams) > 1 and "날짜" in pit.columns:
        pit["날짜_dt"] = pd.to_datetime(pit["날짜"], errors="coerce")
        current_team = pit.sort_values("날짜_dt").iloc[-1]["팀"]
        current_pit = pit[pit["팀"]==current_team]
        if len(current_pit) >= 15:
            pit = current_pit
    result={}
    for side in ["home","away"]:
        s=pit[pit["홈원정"]==side].copy()
        if s.empty: continue
        s["IP_val"]=s["이닝"].apply(ip_to_float); total_ip=s["IP_val"].sum()
        result[side]={"ERA":round((s["자책"].sum()*9)/total_ip,2) if total_ip>0 else 0}
    return result

def get_pitcher_splits(pitcher_name, season=2025):
    try:
        people=requests.get("https://statsapi.mlb.com/api/v1/people/search",params={"names":pitcher_name},timeout=10).json().get("people",[])
        if not people: return {}
        player_id=people[0]["id"]
        res=requests.get(f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats",
            params={"stats":"statSplits","season":season,"group":"pitching","sitCodes":"vl,vr"},timeout=10).json()
        splits=res.get("stats",[{}])[0].get("splits",[])
        result={}
        for s in splits:
            code=s.get("split",{}).get("code",""); stat=s.get("stat",{})
            if code in ["vl","vr"]:
                label="vs좌타" if code=="vl" else "vs우타"
                result[label]={"피안타율":stat.get("avg",""),"OPS":stat.get("ops","")}
        return result
    except: return {}

def get_statcast_pitcher(df_sc_pit, pitcher_name, season=2025):
    row=df_sc_pit[(df_sc_pit["선수명"]==pitcher_name)&(df_sc_pit["시즌"]==season)]
    if row.empty: return {}
    r=row.iloc[0]
    return {"xERA":r.get("xera",""),"ERA_diff":r.get("era_minus_xera_diff",""),
            "피배럴률":r.get("brl_percent",""),"피타구속도":r.get("avg_hit_speed","")}

def get_statcast_batter(df_sc_bat, batter_name, season=2025):
    # [수정] 당해 없으면 3년 누적 PA 500+ 가중평균 활용
    row=df_sc_bat[(df_sc_bat["선수명"]==batter_name)&(df_sc_bat["시즌"]==season)]
    if not row.empty:
        r=row.iloc[0]
        return {"xBA":r.get("est_ba",""),"배럴률":r.get("brl_percent","")}
    hist = df_sc_bat[df_sc_bat["선수명"]==batter_name]
    if not hist.empty:
        total_pa = hist["pa"].sum()
        if total_pa >= 500:
            w_xba = round((hist["est_ba"] * hist["pa"]).sum() / total_pa, 3)
            w_brl = round((hist["brl_percent"] * hist["pa"]).sum() / total_pa, 1)
            return {"xBA":w_xba,"배럴률":w_brl}
    return {}

def get_team_bullpen_stats(df_pit, df_res, team, before_date, n=20):
    before_date=pd.to_datetime(before_date)
    pks=df_res[((df_res["홈팀"]==team)|(df_res["원정팀"]==team))&(df_res["날짜"]<before_date)].tail(n)["gamePk"].tolist()
    pit=df_pit[(df_pit["팀"]==team)&(df_pit["gamePk"].isin(pks))].copy()
    if "날짜" in pit.columns: pit = pit.drop(columns=["날짜"])
    pit["IP_val"]=pit["이닝"].apply(ip_to_float); bp=pit[pit["IP_val"]<5.0]
    if bp.empty: return {}
    total_ip=bp["IP_val"].sum()
    return {"ERA":round((bp["자책"].sum()*9)/total_ip,2) if total_ip>0 else 0,
            "WHIP":round((bp["피안타"].sum()+bp["볼넷"].sum())/total_ip,2) if total_ip>0 else 0,
            "홀드":int(bp["홀드"].sum()),"세이브":int(bp["세이브"].sum())}

def get_bullpen_usage(df_pit, df_res, team, before_date, days=3):
    try:
        before_date=pd.to_datetime(before_date); start_date=before_date-pd.Timedelta(days=days)
        pit=df_pit[df_pit["팀"]==team].copy()
        if "날짜" in pit.columns: pit = pit.drop(columns=["날짜"])
        pit=pit.merge(df_res[["gamePk","날짜"]],on="gamePk",how="left")
        pit["날짜"]=pd.to_datetime(pit["날짜"]); pit["IP_val"]=pit["이닝"].apply(ip_to_float)
        pit=pit[(pit["날짜"]>=start_date)&(pit["날짜"]<before_date)]
        bp=pit[pit["IP_val"]<5.0]
        return {"최근불펜이닝":round(bp["IP_val"].sum(),1) if not bp.empty else 0,
                "최근불펜등판수":len(bp)}
    except: return {"최근불펜이닝":0,"최근불펜등판수":0}

def get_pitcher_home_away_era(df_pit, pitcher_name):
    ha=get_pitcher_home_away(df_pit, pitcher_name)
    return ha.get("home",{}).get("ERA","N/A"), ha.get("away",{}).get("ERA","N/A")

def get_team_day_of_week(df_results, team, before_date, n=30):
    try:
        before_date=pd.to_datetime(before_date); df=df_results.copy()
        df["날짜"]=pd.to_datetime(df["날짜"],errors="coerce")
        df=df[(df["날짜"]<before_date)&((df["홈팀"]==team)|(df["원정팀"]==team))].tail(n)
        if df.empty: return {"요일":"","승률":"N/A"}
        df["요일"]=df["날짜"].dt.day_name(); today_dow=before_date.day_name()
        df_dow=df[df["요일"]==today_dow]
        if df_dow.empty: return {"요일":today_dow,"승률":"N/A"}
        wins=sum(((df_dow["홈팀"]==team)&(df_dow["홈팀득점"]>df_dow["원정팀득점"]))|
                 ((df_dow["원정팀"]==team)&(df_dow["원정팀득점"]>df_dow["홈팀득점"])))
        return {"요일":today_dow,"승률":f"{wins/len(df_dow):.1%}"}
    except: return {"요일":"","승률":"N/A"}

def get_umpire_tendency(df_ump, df_res, umpire_name):
    # [수정] 리그 평균 대비 편차 + 성향 판정 (50경기+ 기준)
    ump_games=df_ump[(df_ump["심판명"]==umpire_name)&(df_ump["포지션"]=="Home Plate")][["gamePk"]]
    if ump_games.empty: return None
    games=ump_games.merge(df_res,on="gamePk",how="left")
    if games.empty: return None
    avg_runs = round(games["총득점"].mean(),2)
    n_games = len(games)
    reg = df_res[df_res["시리즈"]=="정규시즌"] if "시리즈" in df_res.columns else df_res
    league_avg = round(reg["총득점"].mean(), 2)
    diff = round(avg_runs - league_avg, 2)
    if n_games < 50:
        tendency = "표본부족"
    elif diff >= 1.0:
        tendency = "고득점 성향"
    elif diff <= -1.0:
        tendency = "저득점 성향"
    else:
        tendency = "중립"
    return {"경기수":n_games,"평균총득점":avg_runs,"리그대비":diff,"성향":tendency}

def get_batter_recent_form(df_bat, df_res, batter_name, before_date, n=15):
    # [수정] wOBA 15경기 기반 폼 판정 (타율 대비 17% 안정적)
    before_date=pd.to_datetime(before_date)
    bat=df_bat[df_bat["선수명"]==batter_name].copy()
    if "날짜" in bat.columns: bat = bat.drop(columns=["날짜"])
    bat=bat.merge(df_res[["gamePk","날짜"]],on="gamePk",how="left")
    bat["날짜"]=pd.to_datetime(bat["날짜"])
    bat=bat[bat["날짜"]<before_date].sort_values("날짜",ascending=False).head(n)
    if bat.empty: return {}
    pa=bat["타석"].sum()
    woba_num=(bat["볼넷"].sum()*0.69 + bat["안타"].sum()*0.89 +
              bat["2루타"].sum()*0.27 + bat["3루타"].sum()*0.73 +
              bat["홈런"].sum()*1.21)
    woba=round(woba_num/pa,3) if pa>0 else 0
    if woba>=0.370: form="🔥핫"
    elif woba<=0.280: form="❄️콜드"
    else: form="보통"
    ab=bat["타수"].sum(); h=bat["안타"].sum()
    avg=round(h/ab,3) if ab>0 else 0
    return {"경기수":len(bat),"AVG":avg,"wOBA":woba,"홈런":int(bat["홈런"].sum()),"폼":form}

def get_il_players(df_trans, team, before_date, days=30):
    try:
        before_date=pd.to_datetime(before_date); start_date=before_date-pd.Timedelta(days=days)
        df_t=df_trans.copy(); df_t["날짜"]=pd.to_datetime(df_t["날짜"])
        il=df_t[(df_t["팀명"]==team)&(df_t["날짜"]>=start_date)&(df_t["날짜"]<before_date)&
                (df_t["내용"].str.contains("injured list|IL|60-day",case=False,na=False))]
        if il.empty: return "없음"
        # 중복 제거 후 반환
        unique_players = list(dict.fromkeys(il["선수명"].tolist()))
        return " / ".join(unique_players)
    except: return "없음"

def get_pitcher_recent_era(df_pit, df_res, pitcher_name, before_date, days=30):
    # [신규] 투수 최근 30일 ERA (컨디션 지표)
    if not pitcher_name or pitcher_name == "미정": return {}
    try:
        before_date = pd.to_datetime(before_date)
        start_date = before_date - pd.Timedelta(days=days)
        pit = df_pit[df_pit["선수명"]==pitcher_name].copy()
        if "날짜" in pit.columns:
            pit = pit.drop(columns=["날짜"])
        pit = pit.merge(df_res[["gamePk","날짜","시리즈"]], on="gamePk", how="left")
        pit["날짜"] = pd.to_datetime(pit["날짜"])
        # 정규시즌만
        pit = pit[
            (pit["시리즈"]=="정규시즌") &
            (pit["날짜"] >= start_date) &
            (pit["날짜"] < before_date)
        ]
        if pit.empty: return {}
        pit["IP_val"] = pit["이닝"].apply(ip_to_float)
        total_ip = pit["IP_val"].sum()
        if total_ip == 0: return {}
        era_30 = round((pit["자책"].sum()*9)/total_ip, 2)
        return {"ERA_30일": era_30, "등판수": len(pit), "이닝": round(total_ip, 1)}
    except: return {}

def get_pitcher_spring_era(df_spring, pitcher_name, prev_era=None):
    # [신규] 시범경기 토탈 ERA + 전년도 대비 컨디션 판정
    if df_spring is None or df_spring.empty: return {}
    row = df_spring[df_spring["선수명"]==pitcher_name]
    if row.empty: return {}
    r = row.iloc[0]
    spring_era = r.get("spring_ERA", None)
    spring_starts = r.get("spring_등판수", 0)
    spring_ip = r.get("spring_이닝", 0)
    if not spring_era or spring_starts == 0: return {}
    # NaN 방어 처리
    try:
        _starts = int(spring_starts) if spring_starts == spring_starts else 0
        _ip = round(float(spring_ip), 1) if spring_ip == spring_ip else 0.0
        _era = round(float(spring_era), 2)
    except (ValueError, TypeError):
        return {}
    result = {
        "ERA": _era,
        "등판수": _starts,
        "이닝": _ip,
    }
    if prev_era:
        try:
            diff = round(float(spring_era) - float(prev_era), 2)
            result["전년도ERA"] = float(prev_era)
            result["ERA차이"] = diff
            if diff >= 2.0:
                result["컨디션"] = f"⚠️이상(+{diff:.2f})"
            elif diff >= 1.0:
                result["컨디션"] = f"🟡주의(+{diff:.2f})"
            else:
                result["컨디션"] = "✅정상"
        except: pass
    return result

def get_il_players_from_csv(df_il_today, team):
    # [신규] IL현황.csv 기반 IL 조회 (API 직접 수집)
    if df_il_today is None or df_il_today.empty:
        return "없음"
    team_il = df_il_today[df_il_today["팀명"]==team]
    if team_il.empty:
        return "없음"
    il_list = [f"{r['선수명']}({r['IL종류'].replace('Injured ','')})" for _,r in team_il.iterrows()]
    return " / ".join(il_list)

def get_recent_key_players(df_kp, team, before_date, days=3):
    try:
        before_date=pd.to_datetime(before_date); start_date=before_date-pd.Timedelta(days=days)
        _kp=df_kp.copy()
        if "날짜" in _kp.columns: _kp = _kp.drop(columns=["날짜"])
        df_kp_with_date=_kp.merge(df_results[["gamePk","날짜","홈팀","원정팀"]],on="gamePk",how="left")
        df_kp_with_date["날짜"]=pd.to_datetime(df_kp_with_date["날짜"])
        recent=df_kp_with_date[(df_kp_with_date["팀"]==team)&(df_kp_with_date["날짜"]>=start_date)&
            (df_kp_with_date["날짜"]<before_date)&(df_kp_with_date["타입"]=="hitter")].sort_values("게임스코어",ascending=False).head(3)
        if recent.empty: return "없음"
        return " / ".join([f"{r['선수명']}({r['안타']}안타 {r['홈런']}홈런 {r['타점']}타점)" for _,r in recent.iterrows()])
    except: return "없음"

def get_key_bullpen_arms(df_pit, df_res, team, before_date, n_games=20, top_n=3):
    before_date=pd.to_datetime(before_date)
    pks=df_res[((df_res["홈팀"]==team)|(df_res["원정팀"]==team))&(df_res["날짜"]<before_date)].tail(n_games)["gamePk"].tolist()
    pit=df_pit[(df_pit["팀"]==team)&(df_pit["gamePk"].isin(pks))].copy()
    if "날짜" in pit.columns: pit = pit.drop(columns=["날짜"])
    pit["IP_val"]=pit["이닝"].apply(ip_to_float)
    bp=pit[(pit["IP_val"]<5.0)&(pit["IP_val"]>0)]
    if bp.empty: return []
    arm_stats=bp.groupby("선수명").agg(
        등판수=("gamePk","count"),총이닝=("IP_val","sum"),
        총자책=("자책","sum"),세이브=("세이브","sum"),홀드=("홀드","sum")
    ).reset_index()
    arm_stats["ERA_r"]=round((arm_stats["총자책"]*9)/arm_stats["총이닝"],2).where(arm_stats["총이닝"]>0,0)
    arm_stats["중요도"]=arm_stats["세이브"]+arm_stats["홀드"]
    result=[]
    for _,row in arm_stats.sort_values(["중요도","등판수"],ascending=False).head(top_n).iterrows():
        r5=bp[bp["선수명"]==row["선수명"]].tail(5)
        r5ip=r5["IP_val"].sum(); r5er=r5["자책"].sum()
        result.append({"선수명":row["선수명"],
            "역할":"SV" if row["세이브"]>0 else ("HD" if row["홀드"]>0 else "BP"),
            "최근5ERA":round((r5er*9)/r5ip,2) if r5ip>0 else 0.0,
            "최근5이닝":round(r5ip,1),"시즌ERA":row["ERA_r"]})
    return result

def get_matchup_stats(df_matchup, df_pitch, batter_name, pitcher_name, pitcher_hand, batter_id=None):
    if not batter_name or not pitcher_name or pitcher_name=="미정": return {}
    if not df_matchup.empty:
        row=df_matchup[(df_matchup["타자"]==batter_name)&(df_matchup["투수"]==pitcher_name)]
        if not row.empty:
            r=row.iloc[0]; pa=int(r.get("타석",0))
            if pa<5: return {"출처":"직접대결(5타석미만-제외)","타석":pa}
            return {"출처":"직접대결","타석":pa,"AVG":r.get("AVG",""),"홈런":int(r.get("홈런",0))}
    if batter_id:
        try:
            res=requests.get(f"https://statsapi.mlb.com/api/v1/people/{batter_id}/stats",
                params={"stats":"careerStatSplits","group":"hitting","sitCodes":"vr,vl"},timeout=10).json()
            splits=res.get("stats",[{}])[0].get("splits",[])
            code="vr" if pitcher_hand=="R" else "vl"
            for s in splits:
                if s.get("split",{}).get("code")==code:
                    stat=s["stat"]
                    return {"출처":f"커리어 vs {'우투' if pitcher_hand=='R' else '좌투'}",
                            "타석":stat.get("plateAppearances",0),"AVG":stat.get("avg",""),"OPS":stat.get("ops","")}
        except: pass
    return {"출처":"데이터없음"}

def get_team_day_of_week(df_results, team, before_date, n=30): return {'요일':'','승률':'N/A'}  # 제거됨(표본부족)


In [ ]:
# =============================================
# ★ v10 핵심 보조 함수 (정규시즌)
# =============================================

SOURCE_RELIABILITY = {
    "MLB 2026시즌": 10, "MLB 2025시즌": 9,
    "MLB 2024시즌": 6, "MLB 2023시즌": 4,
    "AAA": 4, "AA": 2, "A+": 1, "A": 1, "루키": 1,
    "데이터없음": 0,
}

def get_source_reliability(source_str: str) -> int:
    if not source_str: return 0
    for key, score in SOURCE_RELIABILITY.items():
        if key in source_str:
            return score
    return 3

def classify_lineup_mlb_ratio(df_lineup, game_pk, side):
    lineup = df_lineup[(df_lineup["gamePk"]==game_pk)&(df_lineup["홈원정"]==side)]
    if lineup.empty: return 0.0, 0, 0
    if "레벨" in lineup.columns:
        mlb_cnt = lineup["레벨"].isin(["MLB","AAA"]).sum()
    else:
        mlb_cnt = 0
        for _, row in lineup.iterrows():
            bat_s = get_batter_full_stats(df_player_hit, row["선수명"], df_minor_hit=df_minor_hit)
            src = bat_s.get("출처","")
            if "MLB" in src or "AAA" in src:
                mlb_cnt += 1
    total = len(lineup)
    return round(mlb_cnt/total, 2) if total > 0 else 0.0, mlb_cnt, total

def get_auto_trust_level(home_src, away_src, home_mlb_ratio, away_mlb_ratio,
                          bp_home_era, bp_away_era):
    score = 10.0
    home_rel = get_source_reliability(home_src)
    away_rel  = get_source_reliability(away_src)
    score -= (10 - home_rel) * 0.25
    score -= (10 - away_rel) * 0.25
    if home_mlb_ratio < 0.5 or away_mlb_ratio < 0.5:
        score -= 2.0
    elif home_mlb_ratio < 0.65 or away_mlb_ratio < 0.65:
        score -= 1.0
    if score >= 7.0: return "높음"
    elif score >= 4.5: return "중간"
    else: return "낮음"

def get_run_total_baseline(away_team, home_team, df_run_total_baselines, manual_run_total_baselines):
    key=f"{away_team}@{home_team}"
    if not df_run_total_baselines.empty:
        row=df_run_total_baselines[(df_run_total_baselines["원정팀"]==away_team)&(df_run_total_baselines["홈팀"]==home_team)]
        if not row.empty and pd.notna(row.iloc[0].get("총득점기준점")):
            return float(row.iloc[0]["총득점기준점"]),True
    if key in manual_run_total_baselines: return manual_run_total_baselines[key],True
    return None,False

def classify_il_impact(il_str):
    if not il_str or il_str == "없음":
        return "오늘 경기 영향 없음"
    return f"{il_str} (⚠️ HIGH/MED 영향도는 직접 확인 필요)"

def get_expected_innings(pitcher_name, recent_stats, df_pit_prev=None, season_games=0):
    # [교체] 전년도 정규시즌 기반 + 개막할인율 + QS비율 보정
    prev_avg_ip = None; qs_rate = None
    if df_pit_prev is not None and pitcher_name and pitcher_name != "미정":
        try:
            pit = df_pit_prev[df_pit_prev["선수명"]==pitcher_name].copy()
            if not pit.empty:
                if "IP_val" not in pit.columns:
                    pit["IP_val"] = pit["이닝"].apply(ip_to_float)
                starts = pit[pit["IP_val"] >= 3.0]
                if len(starts) >= 5:
                    prev_avg_ip = round(starts["IP_val"].mean(), 2)
                    qs_count = ((starts["IP_val"] >= 6.0) & (starts["자책"] <= 3)).sum()
                    qs_rate = qs_count / len(starts)
        except: pass
    if prev_avg_ip is not None:
        # 개막 할인율 (실측: 개막일 5.33이닝 / 시즌평균 5.66이닝 = 0.942)
        if season_games <= 7:
            discount = 0.94; d_label = "개막할인x0.94"
        elif season_games <= 14:
            discount = 0.97; d_label = "초반할인x0.97"
        else:
            discount = 1.00; d_label = ""
        # QS비율 보정
        if qs_rate is not None:
            if qs_rate >= 0.60: qs_mult = 1.00; q_label = ""
            elif qs_rate >= 0.40: qs_mult = 0.95; q_label = f"QS{qs_rate:.0%}x0.95"
            elif qs_rate >= 0.20: qs_mult = 0.90; q_label = f"QS{qs_rate:.0%}x0.90"
            else: qs_mult = 0.85; q_label = f"QS{qs_rate:.0%}x0.85"
        else:
            qs_mult = 1.00; q_label = ""
        exp_ip = round(prev_avg_ip * discount * qs_mult, 2)
        labels = [l for l in [f"전년도평균{prev_avg_ip}이닝", d_label, q_label] if l]
        return exp_ip, " ".join(labels), "전년도기반"
    # 전년도 없음: 시범경기 사용 (신인/콜업)
    rip = recent_stats.get("평균이닝", None); ctx = recent_stats.get("출처", "")
    if rip and rip > 0:
        return rip, f"시범경기평균{rip}이닝({ctx})", "시범경기기반"
    return 5.0, "기본값(데이터없음)", "기본값"

def get_risk_tags(home_src, away_src, home_mlb_ratio, away_mlb_ratio,
                   bp_home_era, bp_away_era, run_total_market, home_exp_ip=5.5, away_exp_ip=5.5,
                   weather_effect="중립", il_conflict=False):
    tags = []
    risk_count = 0
    # ⚠️데이터출처낮음
    if get_source_reliability(home_src) <= 2 or get_source_reliability(away_src) <= 2:
        tags.append("⚠️데이터출처낮음"); risk_count += 1
    # ⚠️시장기준선미확보
    if not run_total_market:
        tags.append("⚠️시장기준선미확보"); risk_count += 1
    # ⚠️불펜데이/오프너
    if home_exp_ip <= 2.0 or away_exp_ip <= 2.0:
        tags.append("⚠️불펜데이/오프너"); risk_count += 1
    # ⚠️기상변수큼
    if weather_effect and ("강편향" in str(weather_effect)):
        tags.append("⚠️기상변수큼"); risk_count += 1
    # ⚠️라인업충돌
    if il_conflict:
        tags.append("⚠️라인업충돌"); risk_count += 1
    tag_str = " | ".join(tags) if tags else "없음"
    selection_blocked = risk_count >= 3
    return tag_str, selection_blocked

def get_game_weather(game_pk, df_weather):
    if df_weather.empty: return {}
    row=df_weather[df_weather["gamePk"]==game_pk]
    if row.empty: return {}
    r=row.iloc[0]
    if r.get("돔구장",False): return {"is_dome":True,"str":"돔구장(중립)"}
    ws=r.get("바람속도_mph",0); wd=r.get("바람방향","")
    try: ws=float(ws)
    except: ws=0
    if ws>=12:
        out_d=["N","NNE","NE","ENE","E","S","SSE","SE","ESE"]
        in_d=["NW","NNW","W","WNW","SW","SSW"]
        we="상회_강편향(+0.8pt)" if wd in out_d else ("하회_강편향(-0.8pt)" if wd in in_d else "상회_약편향(+0.4pt)")
    else: we="중립"
    return {"is_dome":False,"str":f"{r.get('기온_F','?')}F | 바람{ws}mph {wd} | {we} | {r.get('날씨','')}","바람효과":we}


In [ ]:
# =============================================
# 프롬프트 정적 섹션 (정규시즌 v10)
# =============================================

PROMPT_STATIC_HEADER = """너는 MLB 경기 자료 분석가다.
15년 경력의 퀀트 애널리스트 + 스카우터 역량 보유.

[절대 원칙]
1. 수치 단순 나열 금지 → 수치 간 관계/패턴에서 인사이트 도출
2. 근거 3가지는 반드시 서로 독립 변수 (중복 카운트 = 원칙 위반)
3. 확률은 반드시 숫자(%)로 명시. "홈팀 유리" 단독 표현 금지
4. 데이터 vs 최신 뉴스 충돌 시 → 최신 뉴스 우선 (보정 이유 명시)
5. 데이터/뉴스 없으면 "모름" 처리. 억측 금지
6. 확률 계산은 반드시 아래 WIN_PROB_FORMULA 단계를 따를 것

[분석 철학 - 최상단 고정]
★ 승패: 내 데이터만으로 예측. 시장 승패 시장가격 사용 안 함
★ 총득점: 시장 총득점 기준점만 사용 (없으면 총득점만 PASS)
★ 점수차: 데이터 내 예상점수차로 독립 판단. 시장 점수차 사용 안 함
★ PASS: 예측 생략이 아닌 판단 보류. 예측값은 항상 출력한다"""

WIN_PROB_FORMULA = """
[WIN_PROB_FORMULA - 반드시 이 공식으로 계산하고 단계를 명시할 것]
BASE = 50.0%

STEP1: 선발 투수 엣지
  유효ERA는 아래 개막전/시즌초반 보강 규칙에 따라 계산
  데이터출처 AA이하: 가중치 0.4배 적용
  ERA차이 1.0당 → ±4% 조정
  예상이닝 구간별 가중치:
    5.1이닝+: 정상
    4.1~5.0이닝: 이 스텝 가중치 80%
    4.0이하: 이 스텝 가중치 60%
    [STEP1 유효ERA 정의 - 개막전/시즌초반 보강]
- MLB현시즌 xERA가 있으면 최우선 사용
- MLB현시즌 xERA가 없고 전시즌 xERA와 ERA가 모두 있으면:
  개막 후 4경기 이내에 한해 유효ERA = (전시즌ERA × 0.5) + (전시즌xERA × 0.5)
- 전시즌 xERA가 없으면 ERA만 사용
- R7 시범경기 컨디션 보정은 위 유효ERA 계산 후 추가 보정한다.

STEP2: 불펜 엣지
  [우선순위: 고레버리지 가용성 > 최근3일 총소모 > 팀불펜ERA]
  1순위: 고레버리지 불펜 3명 가용성 (PHASE1 검색 필수)
    전날 투구수 / 2~3연투 여부 / 오늘 등판 가능 여부
    핵심 마무리/셋업 사용불가 확인 시 → 해당팀 -3%
  2순위: 불펜소모도 기반 개인별 판정
    마무리/셋업 고위험 2명+ → 해당팀 -3%
    마무리/셋업 고위험 1명 → 해당팀 -1.5%
    팀 전체 3일 5이닝+ → 추가 -1%
  3순위: 팀 불펜ERA 차이 1.0당 → ±2% 조정
  ※ 개막 7경기 이내: 팀불펜ERA 신뢰도 낮음 → 3순위 가중치 50%로 축소
  [STEP2 중복 적용 방지]
- 1순위에서 '사용불가'로 판정된 마무리/셋업은 2순위 개인위험 페널티에 중복 포함하지 않는다.
- 1순위 판정 이후 2순위에서는 '팀 전체 최근3일 소모 페널티'만 추가 적용 가능하다.
- 동일 불펜 이슈에 대해 개인위험 페널티를 2회 이상 중복 적용하지 않는다.

STEP3: 타선 엣지
  최근10경기 평균득점 차이 0.5점당 ±1.5%
  팀경기수별 반영 비율:
    0~6경기: 30% 반영 → 대신 확정타순 질로 판단 (아래 보조지표 활용)
    7~14경기: 70% 반영
    15경기+: 100% 반영
  [시즌초반 타선 보조지표 - 팀경기수 0~6경기 시 필수 활용]
    ① 1~6번 타순 기준 vs 선발 손/구종 적합도
    ② 3~5번 핵심타순 최근 wOBA 폼 (🔥핫/❄️콜드)
    ③ 라인업 MLB급 비율 및 상위타순 출루/장타 밸런스
    → 위 3개 지표로 팀득점 보조 판단. 수치화 불가 시 중립(0%) 처리

  [시즌초반 타선 보조지표 점수화 규칙 - 팀경기수 0~6경기]
  각 지표별 우세/중립/열세 판단 후 합산:
    우세 3개: +1.5%
    우세 2개: +1.0%
    우세 1개: +0.5%
    중립(판단불가): 0%
    열세 1개: -0.5%
    열세 2개: -1.0%
    열세 3개: -1.5%
  ※ 최근10경기 득점 기반 30% 반영값과 별도 합산
  ※ STEP3 총 절대값은 최대 3.0% 제한
  [STEP3 시즌초반 보조지표 fallback]
- 최근 wOBA 데이터가 없으면 최근15경기 AVG + OPS로 대체 판단한다.
- 타자별 구종 대응 데이터가 없으면 선발 손잡이 적합도만 사용한다.
- 손잡이 적합도와 구종 대응 데이터가 모두 없으면 해당 지표는 중립(0%) 처리한다.
- vs선발 표본이 5타석 미만이면 대결전적 지표는 사용하지 않는다.

STEP4: 홈/원정 어드밴티지
  파크팩터 95이하 (투수친화): +3.5% 홈
  파크팩터 95~105 (중립):    +3.0% 홈
  파크팩터 105이상 (타자친화): +2.5% 홈

STEP5: 상대전적
  15경기+: 승률 60%+ → ±2%
  5~14경기: 승률 60%+ → ±1% (표본 축소)
  5경기 미만: 제외
  ※ 팀 경기수 20경기 미만이면 STEP5 계산 결과의 50%만 최종 반영한다. (R6 연동)

FINAL = BASE + STEP1~5
범위 강제: 20% ~ 80%
계산 과정 단계별로 명시 필수"""

RUN_TOTAL_FORMULA = """
[RUN_TOTAL_FORMULA - 반드시 이 공식으로 계산하고 단계를 명시할 것]
STEP1: 선발 예상실점
  home_runs = (유효ERA / 9) × 예상이닝
  away_runs = 동일
  ※ RUN_TOTAL_FORMULA의 유효ERA는 WIN_PROB_FORMULA STEP1에서 계산된 최종 유효ERA와 동일하게 사용한다.
  (예상이닝 우선순위: 당일계획 > 최근평균 > 기본값5.5이닝)

STEP2: 불펜 예상실점
  home_bp_ip = 9 - home_starter_ip   ← 홈팀 불펜이 막아야 할 이닝
  away_bp_ip = 9 - away_starter_ip   ← 원정팀 불펜이 막아야 할 이닝
  home_bp_runs = 홈불펜ERA / 9 × home_bp_ip
  away_bp_runs = 원정불펜ERA / 9 × away_bp_ip
  불펜ERA 6.0+ 팀: 해당팀 불펜실점 × 1.3

STEP3: 파크팩터 보정
  (파크팩터 - 100) × 0.03

STEP4: 날씨 보정
  외야 방향 강풍(12mph+): +0.8점
  내야 방향 강풍(12mph+): -0.8점
  돔구장: ±0점
  [구장-날씨 상호작용 보정]
  파크팩터 94 이하 야외구장: 날씨 보정 70%만 반영
    (투수친화 구장은 바람 영향도 상대적으로 억제됨)
  해안/저온 야간구장 + 외야풍: 상회 보정 추가 제한 (-0.1점)
  풍향 신뢰도 낮음(SEARCH_REQUIRED 상태): 날씨 보정 0점 처리

[① 에이스 선발 불펜 노출 억제]
상대팀 선발 유효ERA 3.5 이하 + 예상이닝 5.0이닝 이상인 경우:
→ STEP2 해당팀 불펜 예상실점 계산 시 불펜IP를 30% 축소 적용
  (에이스가 버티면 불펜 노출 자체가 줄어드는 현실 반영)
  예) 원정선발 유효ERA 3.2 + 예상이닝 5.5이닝
    → 홈팀 불펜IP = (9 - 5.5) × 0.7 = 2.45이닝으로 축소

[② 시즌 초반 총득점 하향 보정]
타선 타이밍 부재 + 선발 과잉투구 패턴 반영:
  양팀 경기수 0~3경기: 예상총득점 -0.8점
  양팀 경기수 4~7경기: 예상총득점 -0.4점
  양팀 경기수 8경기+: 보정 없음
※ 양팀 경기수가 서로 다른 경우 평균값 기준 적용

예상총득점 = STEP1(home+away) + STEP2(①억제 적용) + STEP3 + STEP4 + ②시즌초반보정
상회: 예상총득점 > 기준선+0.3
하회: 예상총득점 < 기준선-0.3
판단보류: 그 사이"""

REGULAR_SEASON_RULES = """
[정규시즌 전용 규칙]
R1: 선발 출처 신뢰도 계층: MLB현시즌 > MLB전시즌 > AAA > MLB2년전 > AA이하
    AA이하 선발: 선발엣지 신뢰도 "낮음" 태그
    데이터없음: 선발엣지 제외, 불펜/타선 중심 분석
R2: 예상이닝이 4.0이하 → "단기선발" 태그, 불펜ERA 가중치 1.5배
R3: 시장기준선 없으면 → 총득점 PASS (총득점만 해당, 승패/점수차는 계속 분석)

R4: [PASS 정의 - 판단 보류, 예측 출력은 항상]
    - PASS는 예측 생략이 아닌 판단 보류를 의미한다
    - 모든 경기에서 승패/총득점/점수차 예측은 항상 출력한다
    - 총득점 PASS: 시장기준선 미확보 → 총득점 SELECTION 보류, 예측값 출력은 유지
    - 전체 SELECTION 보류: ⛔위험태그 3개+ → 전체 항목 PASS 태그, 예측값은 출력

R5: [라인업-IL 충돌 우선순위]
    확정라인업 > 구단 공식발표 > 현장기자 트위터 > projected lineup > IL 목록
    라인업과 IL이 충돌하면 → ⚠️라인업충돌 태그 부여 후 재확인 필요
    미해소 시 → 해당 경기 신뢰도 1단계 하향

[라인업충돌 자동 반영 - 강제]
    - 경기 메타 생성 시, 라인업 명단과 IL 명단에 동일 선수가 동시에 존재하면 자동으로 ⚠️라인업충돌 태그를 추가한다.
    - ⚠️라인업충돌 태그가 있으면 승패신뢰도는 최대 '중간'까지로 제한한다.
    - 충돌 미해소 시 승패신뢰도/총득점신뢰도/점수차신뢰도는 모두 '낮음' 처리한다.
    - 경기 메타의 위험태그는 반드시 이 규칙을 반영해 실제 충돌 여부와 일치시킨다.

R6: [시즌 초반 보정 - 팀 경기수 기준]
    팀 경기수 = 당해 정규시즌 경기수만 (시범경기 제외)
    팀 경기수 0~6경기: STEP3 타선엣지 30%만 반영 → 확정타순 보조지표 활용
    팀 경기수 7~14경기: STEP3 타선엣지 70% 반영
    팀 경기수 20경기 미만: STEP5 상대전적 보정 50%로 축소
    개막 후 7경기 이내 + 선발운영계획 미확인:
      승패신뢰도 최대 중간 (높음 불가)
      총득점신뢰도 최대 중간
      점수차신뢰도 최대 낮음
    양팀 선발 모두 운영미확인 + 예상점수차 1.0 미만:
      점수차 자동 PASS
    개막 후 7경기 이내 + 불펜ERA만 존재(소모도 없음):
      STEP2 불펜엣지 가중치 50% 축소

R7: [개막전 선발 컨디션 보정 - 시범경기 반영 규칙]

기본원칙:
- 시범경기 성적/컨디션 정보는 항상 표시한다.
- STEP1 선발엣지에 대한 직접 수치 보정은 선발의 정규시즌 등판 초반에만 적용한다.
- 시범경기 ERA 단독보다 표본/이닝/구속/볼넷/피안타 흐름, 운영계획 코멘트를 우선한다.
- 선발 정규시즌 등판 수 = 최근30일 필드의 등판수 참조.
  데이터없음이면 팀 경기수 기준으로 대체.

[적용 기간 - 선발 개인 등판 수 기준]
- 정규시즌 1~2번째 등판: 시범경기 컨디션 보정 강반영 (100%)
- 정규시즌 3~4번째 등판: 시범경기 컨디션 보정 약반영 (50%)
- 정규시즌 5번째 등판~: 보정 종료, 표시만 유지

[적용 조건]
- 최소 표본: 시범경기 2경기 이상 + 7.0이닝 이상
- 표본 미달 시 보정 절반만 반영
- 운영계획/투구수 제한 뉴스가 있으면 해당 정보가 시범ERA보다 우선

[STEP1 추가 보정 수치]
적용 대상: 정규시즌 1~4번째 등판 선발만

- 시범ERA가 전년도ERA 대비 +3.0 ~ +4.49:
  1~2번째 등판: -1.5%
  3~4번째 등판: -0.75%

- 시범ERA가 전년도ERA 대비 +4.5 ~ +6.9:
  1~2번째 등판: -2.5%
  3~4번째 등판: -1.25%

- 시범ERA가 전년도ERA 대비 +7.0 ~ +9.9:
  1~2번째 등판: -3.5%
  3~4번째 등판: -1.75%

- 시범ERA가 전년도ERA 대비 +10.0 이상:
  1~2번째 등판: -4.5%
  3~4번째 등판: -2.25%

- 시범ERA가 전년도ERA 대비 -2.0 이하이고
  구속/삼진/볼넷 흐름도 정상 이상이면:
  1~2번째 등판: +1.0%
  3~4번째 등판: +0.5%

※ 총 시범경기 컨디션 보정 절대값은 최대 4.5% 제한
※ 동일 구장 장기 강점 / 정상 구속 회복 / 마지막 시범등판 반등 확인 시 보정 50% 축소

R8: 핵심결장 보정
- MVP급/팀 에이스/주전포수/주전유격수 결장 확인 시:
  승패신뢰도 최대 중간
  해당팀 STEP3(타자결장) 또는 STEP1(선발결장) 추가 -1.5%~-3%
- 주전 2명 이상 동시 결장:
  승패/총득점/점수차 신뢰도 모두 1단계 하향
- 결장 정보 미확정(예상 only):
  ⚠️핵심결장 태그만 부여, 수치 보정 없음

[PHASE1 미이행 제한 규칙]
- PHASE1 필수 검색 항목(선발운영계획/고레버리지불펜/날씨) 중 미확인이 2개 이상:
  SELECTION상태 자동 PASS (예측값 출력은 유지)
- 선발운영미확인 + 홈/원정 라인업 미확정 동시 발생:
  승패신뢰도 자동 낮음
- 선발운영미확인 + 고레버리지 불펜 미확인 동시 발생:
  점수차신뢰도 강제 낮음

[위험태그 표준사전 - 아래 7개만 사용]
  ⚠️선발운영미확인  : 예상이닝/등판계획 미확인
  ⚠️시장기준선미확보 : 총득점 기준점 없음
  ⚠️핵심결장        : 주전 or 선발 결장 확인
  ⚠️데이터출처낮음   : AA이하 선발 또는 MLB경기수 부족
  ⚠️기상변수큼       : 12mph+ 강풍 or 강우 예보
  ⚠️불펜데이/오프너  : 선발 2이닝 이하 예상
  ⚠️라인업충돌       : 라인업-IL 정보 불일치
  태그 3개+ → SELECTION상태 필드에 ⛔ 전체 SELECTION 보류 표시 (위험태그 필드에는 ⚠️만 표시)
  ※ 경기 메타의 '⚠️시즌초반' 표기는 설명용 텍스트로만 사용한다.
   위험태그 필드에 포함하지 않으며 SELECTION 보류 판정 개수에 산입하지 않는다.

[신뢰도 분리 원칙 - 승패/총득점/점수차 3개로 독립 판정]
  ※ 경기 메타의 승패사전기본값(분석금지)은 위험태그 반영 전 참고값이며 분석에 직접 사용하지 않는다.
     실제 승패신뢰도는 위험태그와 규칙을 반영해 아래 기준으로 독립 판정한다.
  승패신뢰도:
    높음: 선발운영확인 + 라인업상태(확정) + ⚠️라인업충돌 없음 + 위험태그 1개 이하
    중간: 위 조건 1개 누락 OR 위험태그 2개 OR 라인업상태 미확인
    낮음: 조건 2개 이상 누락 OR ⚠️라인업충돌 미해소
    ※ 라인업상태는 경기 메타의 홈/원정라인업상태 필드 기준
  총득점신뢰도:
    높음: 승패신뢰도 높음 + 시장 총득점 기준선 확보
    중간: 승패신뢰도 중간 + 시장 총득점 기준선 확보
    낮음: 기준선 미확보 OR 승패신뢰도 낮음 → 총득점 PASS
  점수차신뢰도:
    높음: 승패신뢰도 높음 + 예상점수차 1.5점 이상
    중간: (승패신뢰도 높음 OR 승패신뢰도 중간) + 예상점수차 0.5~1.49점
    낮음: 예상점수차 0.5점 미만 OR 데이터 부족 OR 승패신뢰도 낮음
    판단보류: 예상점수차 0.5점 미만 → 점수차 출력하되 SELECTION 불가

[규칙 우선순위]
개별 예외 규칙(R6, 라인업충돌 강제, 선발운영미확인 규칙)이 일반 신뢰도 규칙보다 우선한다.
충돌 시 더 보수적인 판정을 최종값으로 사용한다.
예) R6 → 중간 vs 일반규칙 → 높음 충돌 시 → 중간 채택

[점수차 판단 원칙]
  - 시장 점수차 사용 안 함
  - 데이터 내 예상점수차로 독립 판단
  - 승패/총득점 신뢰도와 무관하게 항상 독립 출력
  - 예상점수차 0.5점 미만: 판단보류
  [점수차 blowout risk 강제 규칙]
  아래 조합 발생 시 점수차신뢰도 강제 낮음 + SELECTION PASS:
    불펜ERA 6.0+ × 예상이닝 4.5이닝 이하 × 선발운영미확인
    (개막전 조기강판/불펜붕괴 복합 리스크 구조)"""

SEARCH_DIRECTIVES = """
[PHASE1: 타겟 검색 - 분석 전 필수 수행]

PRIORITY_A (반드시 검색):
  □ 각 선발의 오늘 예상이닝/운영계획
    검색어: "[투수명] start today innings pitch count"
    미확인 시: 개막 7경기 이내면 신뢰도 자동 하향 적용
  □ 시장기준선 SEARCH_REQUIRED 경기의 실제 총득점 기준점
    검색어: "[팀A] vs [팀B] run total baseline today"
  □ [데이터없음] 선수 최근 실적
    검색어: "[투수명] stats ERA recent"
  □ 고레버리지 불펜 3명 가용성 (개막 7경기 이내 필수)
    검색어: "[팀명] bullpen closer setup available today"
    확인 항목: 전날 등판 여부 / 연투 여부 / 오늘 등판 가능 코멘트

[뉴스 보정 로그] (원본 데이터와 달라진 항목만 기록, 없으면 "없음")
  형식: {항목} / {기존값} → {보정값} / {근거}
  예) 선발 / Logan Webb → TBD / 등판회피 뉴스 확인

PRIORITY_B (중요):
  □ 핵심 선수 부상/결장 뉴스 (최근 48시간)
  □ 로스터 변동 (콜업/DFA/IL 등록)
  □ 선발 투수 컨디션 이슈
  □ 야외구장 날씨 (돔 제외 전 구장 - RUN_TOTAL 보정에 직접 영향)
    검색어: "[구장명] weather forecast today wind"

PRIORITY_C (여유시):
  □ 홈플레이트 심판 성향

[심판 성향 판정 기준]
  ※ 심판 성향은 참고용 변수로만 활용한다.
   RUN_TOTAL_FORMULA 수치 반영 없음. 상회/하회 판단 설명에만 언급 가능.
  출력: "{심판명} | {성향} ({평균총득점}점, 리그대비{diff:+.2f}) | {경기수}경기"
  50경기+: 리그대비 +1.0이상 → 고득점 성향 / -1.0이하 → 저득점 성향 / 그외 → 중립
  중립 심판 → RUN_TOTAL 판단에 영향 없음 (언급 최소화)"""

OUTPUT_FORMAT = """
[PHASE3: 출력 형식 - 경기당 동일 구조 강제 (예측은 항상 출력)]

▶ {원정팀} @ {홈팀}
[경기 검증]
  선발 데이터: 홈={홈선발출처}(신뢰도{홈신뢰도}/10) / 원정={원정선발출처}(신뢰도{원정신뢰도}/10)
  구장: {구장} / 파크팩터: {파크팩터}
  예상이닝: 홈={홈이닝}이닝({근거}) / 원정={원정이닝}이닝({근거})
  총득점 기준선: {기준선}
  승패신뢰도: __ / 총득점신뢰도: __ / 점수차신뢰도: __
  위험태그: {위험태그}

[PHASE1 검색 결과 - 아래 3개 필수 출력]
  선발운영계획: 홈={확인/미확인(구체내용)} / 원정={확인/미확인(구체내용)}
  고레버리지 불펜: 홈={정상/1명위험(이름)/2명+위험} / 원정={정상/1명위험(이름)/2명+위험}
  날씨신뢰도: {높음/중간/낮음} (근거: 풍향/풍속 확인 여부)
[뉴스 보정 로그]
  - {항목} / {기존값} → {보정값} / {근거}
  - 변경사항 없으면: 없음
[활용한 추가정보] (PHASE1 검색 결과 요약)

[핵심 변수 3가지]
  ① (선발 매치업 관련)
  ② (불펜/팀 상태 관련)
  ③ (위 두 가지와 독립적인 변수)

[확률 계산 - WIN_PROB_FORMULA 단계 적용]
  BASE: 50.0%
  STEP1 선발엣지: +/-__% (근거)
  STEP2 불펜엣지: +/-__% (근거)
  STEP3 타선엣지: +/-__% (근거)
  STEP4 홈어드밴티지: +/-__%
  STEP5 상대전적: +/-__%
  FINAL: 홈팀 __% / 원정팀 __%

1. 승패
   예측: 홈팀승 / 원정팀승
   확률: 홈 __% / 원정 __%
   근거①: [선발 매치업 - 유효ERA + 예상이닝 반영]
   근거②: [불펜 ERA + 소모도]
   근거③: [독립 변수]
   승패신뢰도: __
   실행: SELECTION / PASS (이유: __)

2. 총득점
   기준점: {기준선}
   [RUN_TOTAL_FORMULA 단계 계산]
   STEP1 선발예상실점(홈/원정): __/__점
   STEP2 불펜예상실점(홈/원정): __/__점 (단기선발여부: __)
   STEP3 파크팩터보정: +/-__점
   STEP4 날씨보정: +/-__점
   예상총득점: __점
   예측: 상회 / 하회 / 판단보류
   총득점신뢰도: __
   실행: SELECTION / PASS (이유: __)

3. 점수차
   예상점수차: __점 (유리한팀: __)
   예측: 홈점수차적중 / 원정점수차적중 / 판단보류
   점수차신뢰도: __
   실행: SELECTION / PASS (이유: __)

4. 주의사항
   ①: (예측 뒤집을 변수)
   ②: (예측 뒤집을 변수)"""

SUMMARY_FORMAT = """
[전체 경기 요약표]
| 경기 | 승패예측 | 승패실행 | 총득점예측 | 총득점실행 | 점수차예측 | 점수차실행 | 승패신뢰도 | 총득점신뢰도 | 점수차신뢰도 | 위험태그 |
|------|---------|---------|---------|---------|---------|---------|---------|---------|---------|---------|

[TOP3 SELECTION 예측]
조건: 신뢰도 높음 + 실행 SELECTION + 위험태그 최소
1순위: [경기] / [항목] / [예측] / [핵심근거] / [신뢰도]
2순위: ...
3순위: ...
해당없으면: "오늘 SELECTION 예측 없음 - 판단 보류 권고"

[전체 SELECTION 보류 경기]
(⛔ 위험태그 3개+ 경기 전부 + 이유, 예측값은 위에 출력됨)

[성과 측정 기록란 - 결과 입력 후 계산]
- 전체 승패 예측 적중률: __/__
- 전체 총득점 예측 적중률: __/__
- 전체 점수차 예측 적중률: __/__
- SELECTION 승패 적중률: __/__
- SELECTION 총득점 적중률: __/__
- SELECTION 점수차 적중률: __/__

[분석 품질]
- 선발 MLB현시즌 비율: _/_명
- 시장기준선 확보: _/_경기
- 승패신뢰도 높음: _/_경기 / 총득점신뢰도 높음: _/_경기 / 점수차신뢰도 높음: _/_경기"""


In [ ]:
def _get_bp_risk_str(team):
    # 불펜소모도.csv에서 팀 위험등급 요약
    if 'df_bp_detail_today' not in globals() or df_bp_detail_today.empty:
        return "데이터없음"
    team_bp = df_bp_detail_today[df_bp_detail_today["팀명"]==team]
    if team_bp.empty: return "정상"
    high = team_bp[team_bp["위험등급"]=="🔴고위험"]
    mid  = team_bp[team_bp["위험등급"]=="🟡중위험"]
    if not high.empty:
        return f"고위험({','.join(high['선수명'].tolist())})"
    elif not mid.empty:
        return f"중위험({','.join(mid['선수명'].tolist())})"
    return "정상"

def _fmt_key_bp(arms, name):
    if not arms: return f"  {name} 핵심불펜: 데이터없음"
    lines = [f"  {name} 핵심불펜:"]
    for a in arms:
        lines.append(f"    [{a['역할']}]{a['선수명']} 최근5등판ERA{a['최근5ERA']}({a['최근5이닝']}이닝) 시즌ERA{a['시즌ERA']}")
    return chr(10).join(lines)
# =============================================
# 메인 프롬프트 생성 함수 (v3)
# =============================================

def generate_mlb_daily_prompt_v4(date, manual_run_total_baselines=None):
    today_str   = pd.to_datetime(date).strftime("%Y%m%d")
    lineup_path = f"{BT_PATH}/mlb_2025_백테스트_경기목록.csv"
    if not os.path.exists(lineup_path):
        print("백테스트 경기목록 없음!")
        return None

    df_today = pd.read_csv(lineup_path)
    if df_today.empty:
        print("오늘 경기 없음")
        return None
    if manual_run_total_baselines is None: manual_run_total_baselines = {}

    league_avg = round(df_results["총득점"].mean(), 1)

    # 백테스트: 2025 심판.csv에서 gamePk 기반 필터
    _ump_all = df_umpires.copy() if not df_umpires.empty else pd.DataFrame()
    df_today_ump = pd.DataFrame()
    # 심판 컬럼 통일 (라인업콜렉터 포맷: 날짜/gamePk/홈팀/심판명/심판ID/포지션)
    if not _ump_all.empty and "gamePk" in _ump_all.columns:
        df_today_ump = _ump_all[_ump_all["gamePk"].isin(df_today["gamePk"].tolist())].copy()
        if not df_today_ump.empty and "시즌" not in df_today_ump.columns:
            df_today_ump["포지션"] = df_today_ump["포지션"].str.replace("Home Plate Umpire","Home Plate").str.replace("First Base Umpire","First Base").str.replace("Second Base Umpire","Second Base").str.replace("Third Base Umpire","Third Base")

    today_lu_path   = f"{BT_PATH}/mlb_2025_백테스트_라인업.csv"
    df_lineup_today = pd.read_csv(today_lu_path) if os.path.exists(today_lu_path) else pd.DataFrame()
    if not df_lineup_today.empty:
        df_lineup_today["날짜"] = pd.to_datetime(df_lineup_today["날짜"]).dt.strftime("%Y-%m-%d")
        df_lineup_today = df_lineup_today[df_lineup_today["날짜"] == BT_DATE]

    # ─── 프롬프트 헤더 ───
    prompt = f"""{PROMPT_STATIC_HEADER}

분석 날짜: {date}  |  대상 경기 수: {len(df_today)}경기

{WIN_PROB_FORMULA}

{RUN_TOTAL_FORMULA}

{REGULAR_SEASON_RULES}

{SEARCH_DIRECTIVES}

{'━'*55}
📋 경기 데이터 블록
{'━'*55}

"""

    skipped_games = []

    for _, game in df_today.iterrows():
        home_team = game["홈팀"]
        away_team = game["원정팀"]
        stadium   = game["구장"]
        game_pk   = game["gamePk"]

        # 라인업 확인
        game_lineup = df_lineup_today[df_lineup_today["gamePk"]==game_pk] if not df_lineup_today.empty else pd.DataFrame()
        if game_lineup.empty:
            skipped_games.append(f"{away_team} @ {home_team} (라인업 미확정)")
            continue

        # ── 파크팩터 ──
        pf, pf_valid = get_park_factor(df_results, stadium)
        pf_tag = "" if pf_valid else "⚠️표본부족"

        # ── 팀 스탯 ──
        home_stats = get_recent_team_stats(df_results, home_team, date)
        away_stats = get_recent_team_stats(df_results, away_team, date)

        # ★ 라인업상태 + IL충돌 감지
        home_lineup_status = "확정" if game_pk in (df_lineup_today["gamePk"].unique() if not df_lineup_today.empty else []) else "미확인"
        away_lineup_status = home_lineup_status  # 같은 경기이므로 동일
        # IL 충돌 체크는 home_il_raw/away_il_raw 정의 후 수행
        il_conflict = False

        # ★ 팀 시즌 경기수 계산 (시즌 초반 보정용)
        _season_year = pd.to_datetime(date).year
        # [수정] R6: 정규시즌 경기수만 카운트 (시범경기 제외)
        _reg_only = df_results[df_results["시리즈"]=="정규시즌"] if "시리즈" in df_results.columns else df_results
        _home_season_games = len(_reg_only[
            (_reg_only["날짜"] >= f"{_season_year}-03-01") &
            (_reg_only["날짜"] < date) &
            ((_reg_only["홈팀"]==home_team)|(_reg_only["원정팀"]==home_team))
        ])
        _away_season_games = len(_reg_only[
            (_reg_only["날짜"] >= f"{_season_year}-03-01") &
            (_reg_only["날짜"] < date) &
            ((_reg_only["홈팀"]==away_team)|(_reg_only["원정팀"]==away_team))
        ])
        home_split = get_home_away_stats(df_results, home_team, date)
        away_split = get_home_away_stats(df_results, away_team, date)
        h2h        = get_head_to_head(df_results, home_team, away_team)

        # ── 선발 ──
        home_starter = game.get("홈팀선발","미정")
        away_starter = game.get("원정팀선발","미정")
        home_season  = get_starter_full_stats(df_player_pit, home_starter, 2024)
        away_season  = get_starter_full_stats(df_player_pit, away_starter, 2024)
        home_recent  = get_pitcher_last_n(df_pitchers, df_results, home_starter, date) or {}
        away_recent  = get_pitcher_last_n(df_pitchers, df_results, away_starter, date) or {}

        # [수정] 예상이닝: 전년도 정규시즌 기반 + 개막할인율 + QS보정
        _df_pit_prev = df_pit_prev if 'df_pit_prev' in globals() else None
        home_exp_ip, home_ip_basis, _ = get_expected_innings(
            home_starter, home_recent, df_pit_prev=_df_pit_prev, season_games=_home_season_games)
        away_exp_ip, away_ip_basis, _ = get_expected_innings(
            away_starter, away_recent, df_pit_prev=_df_pit_prev, season_games=_away_season_games)

        # ★ 불펜전 판정
        bullpen_game_flag = "불펜전" if home_exp_ip <= 3.0 or away_exp_ip <= 3.0 else "일반"

        # ── 불펜 ──
        home_bp      = get_bullpen_usage(df_pitchers, df_results, home_team, date)
        away_bp      = get_bullpen_usage(df_pitchers, df_results, away_team, date)
        home_bp_stats = get_team_bullpen_stats(df_pitchers, df_results, home_team, date)
        away_bp_stats = get_team_bullpen_stats(df_pitchers, df_results, away_team, date)

        # ── MLB 비율 ──
        home_ratio, home_mlb_cnt, home_total = classify_lineup_mlb_ratio(df_lineup_today, game_pk, "home")
        away_ratio, away_mlb_cnt, away_total = classify_lineup_mlb_ratio(df_lineup_today, game_pk, "away")

        # ── 승패기본신뢰도 (선발/라인업/데이터 기준, 시장기준점 제외) ──
        auto_trust = get_auto_trust_level(
            home_season.get("출처",""),
            away_season.get("출처",""),
            home_ratio, away_ratio,
            home_bp_stats.get("ERA","4.0"),
            away_bp_stats.get("ERA","4.0")
        )
        # ① 개막 7경기 이내: 사전기본값 미산정 처리 (규칙이 항상 우선)
        if _home_season_games < 7 or _away_season_games < 7:
            auto_trust = "미산정(개막초반-규칙 우선 적용)"

        # ★ 총득점 기준선
        run_total_val, run_total_ok = get_run_total_baseline(away_team, home_team, df_run_total_baselines, manual_run_total_baselines)
        run_total_baseline = f'{run_total_val} (시장기준)' if run_total_ok else 'SEARCH_REQUIRED -> 총득점 판단보류'
        run_total_is_market = run_total_ok

        # ★ 위험 태그

        # 날씨
        w_data = get_game_weather(game_pk, df_weather)
        weather_str = w_data.get('str','SEARCH_REQUIRED') if w_data else 'SEARCH_REQUIRED'

        # ★ 위험태그는 IL 충돌 감지 후에 계산 (아래로 이동)
        _risk_tags_pending = True

        # 핵심 불펜
        home_key_bp = get_key_bullpen_arms(df_pitchers, df_results, home_team, date)
        away_key_bp = get_key_bullpen_arms(df_pitchers, df_results, away_team, date)

        # ── 나머지 데이터 ──
        home_hand = get_pitcher_hand(home_starter)
        away_hand = get_pitcher_hand(away_starter)
        home_pitch_stats = get_pitcher_pitch_stats(df_pitch, home_starter)
        away_pitch_stats = get_pitcher_pitch_stats(df_pitch, away_starter)
        home_offense = get_team_offense_pattern(df_results, df_batters, home_team, date)
        away_offense = get_team_offense_pattern(df_results, df_batters, away_team, date)
        home_park = get_pitcher_park_stats(df_pitchers, df_results, home_starter, stadium)
        away_park = get_pitcher_park_stats(df_pitchers, df_results, away_starter, stadium)
        home_ha_e, home_ha_a = get_pitcher_home_away_era(df_pitchers, home_starter)
        away_ha_e, away_ha_a = get_pitcher_home_away_era(df_pitchers, away_starter)
        home_splits = get_pitcher_splits(home_starter)
        away_splits = get_pitcher_splits(away_starter)
        home_sc = get_statcast_pitcher(df_sc_pit, home_starter)
        away_sc = get_statcast_pitcher(df_sc_pit, away_starter)
        # [추가] 투수 최근 30일 ERA (컨디션 지표)
        home_era30 = get_pitcher_recent_era(df_pitchers, df_results, home_starter, date)
        away_era30 = get_pitcher_recent_era(df_pitchers, df_results, away_starter, date)
        # [추가] 시범경기 토탈 ERA + 컨디션 판정
        _df_spring = df_spring_stats if "df_spring_stats" in globals() else pd.DataFrame()
        home_spring = get_pitcher_spring_era(_df_spring, home_starter, home_season.get("ERA"))
        away_spring = get_pitcher_spring_era(_df_spring, away_starter, away_season.get("ERA"))
        home_dow = get_team_day_of_week(df_results, home_team, date)
        away_dow = get_team_day_of_week(df_results, away_team, date)

        # ── 심판 ──
        ump_name="미정"; ump_stats={}
        if not df_today_ump.empty:
            ump_row = df_today_ump[(df_today_ump["gamePk"]==game_pk)&(df_today_ump["포지션"]=="Home Plate")]
            if not ump_row.empty:
                ump_name=ump_row.iloc[0]["심판명"]; ump_stats=get_umpire_tendency(df_umpires, df_results, ump_name) or {}

        # [수정] IL: API 직접 수집 IL현황.csv 기반
        _df_il = df_il_today if 'df_il_today' in globals() else pd.DataFrame()
        home_il_raw = get_il_players_from_csv(_df_il, home_team)
        away_il_raw = get_il_players_from_csv(_df_il, away_team)
        home_il = classify_il_impact(home_il_raw)
        away_il = classify_il_impact(away_il_raw)



        # ★ IL 충돌 감지 - 라인업 선수 vs IL 선수 직접 비교
        # (home_il_raw / away_il_raw 정의 후 실행)
        try:
            # 1순위: df_games의 IL충돌선수 컬럼 (라인업 콜렉터가 감지한 경우)
            if not df_lineup_today.empty and "IL충돌선수" in df_today.columns:
                game_row = df_today[df_today["gamePk"] == game_pk]
                if not game_row.empty and pd.notna(game_row.iloc[0].get("IL충돌선수", None)):
                    il_conflict = True
            # 2순위: 라인업 선수 명단 vs IL 선수 명단 직접 비교
            # (get_il_players는 문자열 반환 - "/" 구분자로 분리)
            if not il_conflict and not df_lineup_today.empty:
                game_lu_players = set(
                    df_lineup_today[df_lineup_today["gamePk"] == game_pk]["선수명"].dropna().tolist()
                )
                def _parse_il_str(il_str):
                    if not il_str or il_str in ("없음", ""):
                        return set()
                    return set([p.strip() for p in str(il_str).split("/") if p.strip()])
                il_home_set = _parse_il_str(home_il_raw)
                il_away_set = _parse_il_str(away_il_raw)
                if game_lu_players & (il_home_set | il_away_set):
                    il_conflict = True
        except Exception as _e:
            print(f"IL충돌감지 오류: {_e}")

        # ★ 위험태그 계산 (IL 충돌 감지 후)
        risk_tags, _selection_blocked = get_risk_tags(
            home_season.get("출처",""),
            away_season.get("출처",""),
            home_ratio, away_ratio,
            home_bp_stats.get("ERA","4.0"),
            away_bp_stats.get("ERA","4.0"),
            run_total_is_market,
            home_exp_ip=home_exp_ip,
            away_exp_ip=away_exp_ip,
            weather_effect=w_data.get("바람효과","중립") if w_data else "중립",
            il_conflict=il_conflict
        )
        # ① 개막초반 PHASE1 전: SELECTION상태 미판정 처리
        if _home_season_games < 7 or _away_season_games < 7:
            selection_status = "⛔ 전체 SELECTION 보류 (위험태그 3개+)" if _selection_blocked else "PHASE1 전/미판정(개막초반)"
        else:
            selection_status = "⛔ 전체 SELECTION 보류 (위험태그 3개+)" if _selection_blocked else "정상"

        # ── 키플레이어 ──
        home_kp = get_recent_key_players(df_kp, home_team, date) if not df_kp.empty else "없음"
        away_kp = get_recent_key_players(df_kp, away_team, date) if not df_kp.empty else "없음"

        # ★ 출처 신뢰도 점수
        home_rel = get_source_reliability(home_season.get("출처",""))
        away_rel  = get_source_reliability(away_season.get("출처",""))

        # ─── 타자 라인업 블록 (홈/원정 분리) ───
        def _make_lu_rows(side, opp_starter, opp_hand):
            rows = []
            side_lu = game_lineup[game_lineup["홈원정"]==side].sort_values("타순")
            if side_lu.empty: return ["    라인업 미확정"]
            for i, (_, row2) in enumerate(side_lu.iterrows()):
                bat_s = get_batter_full_stats(df_player_hit, row2["선수명"], df_minor_hit=df_minor_hit)
                bat_src = _clean_source(bat_s.get("출처","N/A"))
                # 20타석 미만은 OPS 노이즈 → N/A 처리
                _pa = bat_s.get("plateAppearances", bat_s.get("타석", 0))
                try: _pa = int(_pa)
                except: _pa = 0
                bat_ops = "N/A(소수타석)" if _pa < 20 else _clean_ops(bat_s.get("ops", bat_s.get("OPS","N/A")))
                bat_avg = _clean_avg(bat_s.get("avg", bat_s.get("AVG","N/A")))
                sc_b  = get_statcast_batter(df_sc_bat, row2["선수명"])
                rf    = get_batter_recent_form(df_batters, df_results, row2["선수명"], date)
                mu    = get_matchup_stats(df_matchup, df_pitch, row2["선수명"], opp_starter, opp_hand, row2.get("선수ID"))
                if "5타석미만" in mu.get("출처",""): mu_str = "vs선발:[5타석미만제외]"
                elif mu.get("출처","")=="데이터없음": mu_str = "vs선발:[데이터없음]"
                else: mu_str = f"vs선발[{mu.get('출처','')}] {mu.get('타석',0)}타석 AVG{mu.get('AVG','N/A')}"
                rows.append(
                    f"    {i+1}번 {row2['포지션']} {row2['선수명']} [{bat_src}] "
                    f"AVG{bat_avg} OPS{bat_ops} HR{bat_s.get('홈런','N/A')} | "
                    f"xBA{sc_b.get('xBA','N/A')} 배럴{sc_b.get('배럴률','N/A')}% | "
                    f"최근15경기{rf.get('AVG','N/A')}{rf.get('폼','')} | {mu_str}"
                )
            return rows
        home_lu = _make_lu_rows("home", away_starter, away_hand)
        away_lu = _make_lu_rows("away", home_starter, home_hand)
        lineup_block = (
            f"  [홈] {home_team}\n" + "\n".join(home_lu) + "\n" +
            f"  [원정] {away_team}\n" + "\n".join(away_lu)
        )

        prompt += f"""{('━'*55)}
        ⚾ {away_team} @ {home_team}
{('━'*55)}
[경기 메타 - 자동 계산값]
  구장: {stadium} | 파크팩터: {pf}{f" ({pf_tag})" if pf_tag else ""}
  팀경기수: 홈 {_home_season_games}경기 / 원정 {_away_season_games}경기{"  ⚠️시즌초반" if _home_season_games < 15 or _away_season_games < 15 else ""}
  선발신뢰도: 홈 {home_rel}/10 ({home_season.get('출처','N/A')}) / 원정 {away_rel}/10 ({away_season.get('출처','N/A')})
  홈MLB비율: {home_mlb_cnt}/{home_total}명({home_ratio:.0%}) / 원정MLB비율: {away_mlb_cnt}/{away_total}명({away_ratio:.0%})
  예상이닝: 홈 {home_exp_ip}이닝({home_ip_basis}) / 원정 {away_exp_ip}이닝({away_ip_basis})
  날씨: {weather_str}
  불펜전여부: {bullpen_game_flag}
  홈라인업상태: {home_lineup_status} / 원정라인업상태: {away_lineup_status}
  총득점기준선: {run_total_baseline}
  승패사전기본값(분석금지): {auto_trust}
  위험태그: {risk_tags}
  SELECTION상태: {selection_status}
  심판: {ump_name}{f" | {ump_stats.get('성향','N/A')} ({ump_stats.get('평균총득점','N/A')}점, 리그대비{ump_stats.get('리그대비',0):+.2f}) | {ump_stats.get('경기수','N/A')}경기" if ump_stats else ""}

[선발 투수]
홈: {home_starter} [{home_season.get('출처','N/A')}] 신뢰도{home_rel}/10
  ERA {home_season.get('ERA','N/A')} WHIP {home_season.get('WHIP','N/A')} K/9 {home_season.get('K/9','N/A')} BB/9 {home_season.get('BB/9','N/A')} {home_season.get('승','N/A')}승 {home_season.get('패','N/A')}패 {home_season.get('이닝','')}이닝
  시범경기(2025): {(str(home_spring['등판수'])+'경기 '+str(home_spring['이닝'])+'이닝 ERA '+str(home_spring['ERA'])+' '+home_spring.get('컨디션','')+' (전년도ERA '+str(home_spring.get('전년도ERA','N/A'))+')') if home_spring else '데이터없음'}
  최근3경기({home_recent.get('출처','?')}): 평균 {home_recent.get('평균이닝','N/A')}이닝 {home_recent.get('평균자책','N/A')}자책 {home_recent.get('평균삼진','N/A')}K
  ⭐예상이닝오늘: {home_exp_ip}이닝 ({home_ip_basis})
  최근30일: {f"ERA {home_era30.get('ERA_30일','N/A')} ({home_era30.get('등판수','?')}등판 {home_era30.get('이닝','?')}이닝)" if home_era30 else "정규시즌 데이터없음"}
  Statcast: xERA {home_sc.get('xERA','N/A')} ERA차이 {home_sc.get('ERA_diff','N/A')} | 피배럴률 {home_sc.get('피배럴률','N/A')}% | 피타구속도 {home_sc.get('피타구속도','N/A')}mph
  홈/원정ERA: 홈 {home_ha_e} / 원정 {home_ha_a}
  좌우분리: {f"vs좌타 피안{home_splits.get('vs좌타',{}).get('피안타율','N/A')} OPS{home_splits.get('vs좌타',{}).get('OPS','N/A')} | vs우타 피안{home_splits.get('vs우타',{}).get('피안타율','N/A')} OPS{home_splits.get('vs우타',{}).get('OPS','N/A')}" if home_splits else "데이터없음"}
  구종: {" | ".join([f"{k} {v['사용률']}% {v['평균구속']}mph 헛스윙{v['헛스윙률']}% 피안{v['피안타율']}" for k,v in home_pitch_stats.items()]) if home_pitch_stats else "데이터없음"}
  구장({stadium}): {f"{home_park['등판수']}경기 ERA {home_park['ERA']} WHIP {home_park['WHIP']}" if home_park else "데이터없음"}

원정: {away_starter} [{away_season.get('출처','N/A')}] 신뢰도{away_rel}/10
  ERA {away_season.get('ERA','N/A')} WHIP {away_season.get('WHIP','N/A')} K/9 {away_season.get('K/9','N/A')} BB/9 {away_season.get('BB/9','N/A')} {away_season.get('승','N/A')}승 {away_season.get('패','N/A')}패 {away_season.get('이닝','')}이닝
  시범경기(2025): {(str(away_spring['등판수'])+'경기 '+str(away_spring['이닝'])+'이닝 ERA '+str(away_spring['ERA'])+' '+away_spring.get('컨디션','')+' (전년도ERA '+str(away_spring.get('전년도ERA','N/A'))+')') if away_spring else '데이터없음'}
  최근3경기({away_recent.get('출처','?')}): 평균 {away_recent.get('평균이닝','N/A')}이닝 {away_recent.get('평균자책','N/A')}자책 {away_recent.get('평균삼진','N/A')}K
  ⭐예상이닝오늘: {away_exp_ip}이닝 ({away_ip_basis})
  최근30일: {f"ERA {away_era30.get('ERA_30일','N/A')} ({away_era30.get('등판수','?')}등판 {away_era30.get('이닝','?')}이닝)" if away_era30 else "정규시즌 데이터없음"}
  Statcast: xERA {away_sc.get('xERA','N/A')} ERA차이 {away_sc.get('ERA_diff','N/A')} | 피배럴률 {away_sc.get('피배럴률','N/A')}% | 피타구속도 {away_sc.get('피타구속도','N/A')}mph
  홈/원정ERA: 홈 {away_ha_e} / 원정 {away_ha_a}
  좌우분리: {f"vs좌타 피안{away_splits.get('vs좌타',{}).get('피안타율','N/A')} OPS{away_splits.get('vs좌타',{}).get('OPS','N/A')} | vs우타 피안{away_splits.get('vs우타',{}).get('피안타율','N/A')} OPS{away_splits.get('vs우타',{}).get('OPS','N/A')}" if away_splits else "데이터없음"}
  구종: {" | ".join([f"{k} {v['사용률']}% {v['평균구속']}mph 헛스윙{v['헛스윙률']}% 피안{v['피안타율']}" for k,v in away_pitch_stats.items()]) if away_pitch_stats else "데이터없음"}
  구장({stadium}): {f"{away_park['등판수']}경기 ERA {away_park['ERA']} WHIP {away_park['WHIP']}" if away_park else "데이터없음"}

[팀 불펜]
{home_team}: ERA {home_bp_stats.get('ERA','N/A')} WHIP {home_bp_stats.get('WHIP','N/A')} 홀드{home_bp_stats.get('홀드',0)} 세이브{home_bp_stats.get('세이브',0)} {"🚨ERA6+" if float(home_bp_stats.get('ERA','4') or 4) >= 6.0 else ""}
{away_team}: ERA {away_bp_stats.get('ERA','N/A')} WHIP {away_bp_stats.get('WHIP','N/A')} 홀드{away_bp_stats.get('홀드',0)} 세이브{away_bp_stats.get('세이브',0)} {"🚨ERA6+" if float(away_bp_stats.get('ERA','4') or 4) >= 6.0 else ""}
불펜소모 최근3일: {home_team} {home_bp['최근불펜이닝']}이닝 {home_bp['최근불펜등판수']}등판 | {away_team} {away_bp['최근불펜이닝']}이닝 {away_bp['최근불펜등판수']}등판
불펜위험등급: {home_team} {_get_bp_risk_str(home_team)} | {away_team} {_get_bp_risk_str(away_team)}
{_fmt_key_bp(home_key_bp, home_team)}
{_fmt_key_bp(away_key_bp, away_team)}

[최근 10경기]
{home_team}: {home_stats['승']}승 {home_stats['패']}패 (승률{home_stats['승률']:.1%}) | 평균 {home_stats['평균득점']}득점 {home_stats['평균실점']}실점 | 홈승률{home_split['홈승률']:.1%}
{away_team}: {away_stats['승']}승 {away_stats['패']}패 (승률{away_stats['승률']:.1%}) | 평균 {away_stats['평균득점']}득점 {away_stats['평균실점']}실점 | 원정승률{away_split['원정승률']:.1%}

[상대전적 최근 30경기]
{f"{h2h['총경기']}경기: {home_team} {h2h['홈팀승']}승 {h2h['원정팀승']}패 | 평균총득점 {h2h['평균총득점']}점 | 평균점수차 {h2h['평균점수차']}점" if h2h else "데이터없음"}

[팀 득점 패턴 최근 20경기]
{home_team}: 평균 {home_offense.get('평균득점','N/A')}득점 | 홈런의존도 {home_offense.get('홈런의존도','N/A')}% | 팀타율 {home_offense.get('팀타율','N/A')} | {home_offense.get('득점유형','N/A')}
{away_team}: 평균 {away_offense.get('평균득점','N/A')}득점 | 홈런의존도 {away_offense.get('홈런의존도','N/A')}% | 팀타율 {away_offense.get('팀타율','N/A')} | {away_offense.get('득점유형','N/A')}

[타자 라인업 + 매치업]
  MLB급비율 홈:{home_mlb_cnt}/{home_total}명({home_ratio:.0%}) | 원정:{away_mlb_cnt}/{away_total}명({away_ratio:.0%})
{lineup_block}

[IL - 오늘 영향도 포함]
{home_team}: {home_il}
{away_team}: {away_il}

[최근 활약 키플레이어]
{home_team}: {home_kp}
{away_team}: {away_kp}

"""

    if skipped_games:
        prompt += f"""{('━'*55)}
        ⏸ 라인업 미확정 제외 경기
{('━'*55)}
{chr(10).join([f"  - {g}" for g in skipped_games])}

"""

    prompt += f"""{('━'*55)}
[PHASE2: 분석 실행]
{('━'*55)}
위 데이터 + PHASE1 검색 결과를 종합해서 각 경기를 아래 형식으로 분석하라.

{OUTPUT_FORMAT}

{('━'*55)}
[PHASE4: 전체 요약]
{('━'*55)}
{SUMMARY_FORMAT}
"""
    return prompt

In [ ]:
# Cell 9: 중복 정의 제거
pass


In [ ]:
def get_park_factor(df, stadium):
    reg = df[df["시리즈"]=="정규시즌"] if "시리즈" in df.columns else df
    valid_stadiums = reg.groupby("구장").size()
    valid_stadiums_80 = valid_stadiums[valid_stadiums >= 80].index.tolist()
    reg_valid = reg[reg["구장"].isin(valid_stadiums_80)]
    overall_avg = reg_valid["총득점"].mean() if not reg_valid.empty else reg["총득점"].mean()
    if pd.isna(overall_avg) or overall_avg == 0:
        return 100.0, False
    stadium_data = reg[reg["구장"]==stadium]
    if stadium_data.empty:
        return 100.0, False
    stadium_avg = stadium_data["총득점"].mean()
    if pd.isna(stadium_avg):
        return 100.0, False
    n_games = len(stadium_data)
    pf = round(stadium_avg/overall_avg*100, 1)
    is_valid = n_games >= 80
    return pf, is_valid

print("get_park_factor 재정의 완료")

In [ ]:
# df_minor_hit 컬럼 강제 지정 (2025 백테스트용)
minor_hit_path = f"{MLB_PATH}/2025/mlb_2025_마이너성적_타자.csv"

if os.path.exists(minor_hit_path):
    df_minor_hit = pd.read_csv(minor_hit_path)
    if df_minor_hit.columns[0] != "선수명":
        df_minor_hit = pd.read_csv(minor_hit_path, header=None,
            names=["선수명","선수ID","avg","ops","homeRuns","plateAppearances","hits","atBats","출처","팀"])
    df_minor_hit = df_minor_hit.fillna("")
    print(f"마이너 타자 로드: {len(df_minor_hit)}명")
    print(df_minor_hit.head(3))
else:
    df_minor_hit = pd.DataFrame()
    print("마이너 타자 데이터 없음")


In [ ]:
print(f'프롬프트 생성 중... ({DATE})')
print(f'RUN_TOTAL:{len(df_run_total_baselines)}경기 날씨:{len(df_weather)}경기')
prompt = generate_mlb_daily_prompt_v4(
    date=DATE, manual_run_total_baselines=MANUAL_RUN_TOTAL_BASELINES
)
if prompt:
    print(prompt)
    save_path = f'{BT_PATH}/mlb_{TODAY}_백테스트프롬프트_v11.txt'
    with open(save_path,'w',encoding='utf-8') as f: f.write(prompt)
    print(f'저장: {save_path}')
